<a href="https://colab.research.google.com/github/krishna11-dot/EmergingMarkets-ValueInvestor/blob/main/__stock_prediction_system___.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pygad

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import random
warnings.filterwarnings('ignore')

# For preprocessing and feature engineering
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# For deep learning models
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# For genetic algorithm optimization
import pygad

class StockPredictionSystem:
    """
    A comprehensive system for stock price prediction and trading
    strategy optimization with focus on:
    - Bollinger Bands as baseline strategy (using 20-day window as per industry standard)
    - Enhanced LSTM and Transformer models (with flexible sequence length)
    - Market-specific parameter optimization
    - Visualization-focused performance analysis
    """

    #################################
    # SECTION 1: INITIALIZATION
    #################################
    def __init__(self, file_path):
        """Initialize the prediction system with file path to Excel data"""
        self.file_path = file_path
        self.companies = {}  # Will store data for each company
        self.preprocessed_data = {}  # Will store preprocessed data
        self.featured_data = {}  # Will store data with engineered features
        self.models = {}  # Will store trained models
        self.predictions = {}  # Will store predictions
        self.trading_results = {}  # Will store trading strategy results
        self.visualizations = {}  # Will store key visualizations

        # Default parameters - adjusted for better performance
        self.params = {
            'seq_length': 6,  # Updated from 5 to 6 days as suggested by Bhargav
            'hidden_dim': 64,
            'num_layers': 3,
            'batch_size': 16,
            'num_epochs': 50,
            'learning_rate': 0.001,
            'dropout': 0.2
        }

        # Market-specific parameters
        self.market_params = {
            'South Africa - Impala Platinum': {
                'bollinger_window': 30,  # Extended window for this specific market
                'seq_length': 7,         # Longer sequence for this market
                'strategy': 'enhanced'   # Using enhanced strategy instead of model-based
            }
        }

    #################################
    # SECTION 2: DATA LOADING & PREPROCESSING
    #################################
    def load_data(self, display_samples=True):
        """
        Load multi-sheet Excel file with each sheet containing a company's stock data
        Each company has columns: Date, Price, Open, High, Low, Vol., Change %
        """
        print("\n" + "="*80)
        print("STEP 1: LOADING DATASET")
        print("="*80)
        print("Loading stock data from multiple companies")

        # Load Excel file
        excel_file = pd.ExcelFile(self.file_path)
        sheet_names = excel_file.sheet_names

        print(f"Found {len(sheet_names)} companies in the dataset:")
        for i, sheet in enumerate(sheet_names):
            print(f" {i+1}. {sheet}")

        # Process each company sheet
        for sheet_name in sheet_names:
            print(f"\nProcessing {sheet_name}...")

            # Load the raw data
            df = pd.read_excel(self.file_path, sheet_name=sheet_name)

            # Display sample if needed
            if display_samples:
                print(f"Original shape: {df.shape}")
                print("\nFirst 3 rows:")
                print(df.head(3))

            # Store in the companies dictionary
            self.companies[sheet_name] = {
                'raw_data': df,
                'name': sheet_name
            }

            print(f"Loaded data for {sheet_name}")

        return self.companies

    def preprocess_data(self):
        """
        Preprocess the raw data:
        - Convert dates to datetime
        - Convert prices to numeric
        - Process volume with proper K/M/B handling
        - Add date-related columns for analysis
        """
        print("\n" + "="*80)
        print("STEP 2: PREPROCESSING DATA")
        print("="*80)

        for company_name, company_data in self.companies.items():
            print(f"\nPreprocessing {company_name}...")

            # Get the raw data
            df = company_data['raw_data'].copy()

            # Step 2.1: Remove summary rows (rows with missing Vol. or Change %)
            print(" - Removing summary rows...")
            df = df.dropna(subset=['Vol.', 'Change %'])

            # Step 2.2: Convert dates to datetime
            print(" - Converting dates to datetime format...")
            df['Date'] = pd.to_datetime(df['Date'])

            # Step 2.3: Convert price columns to numeric
            print(" - Converting price columns to numeric...")
            for col in ['Price', 'Open', 'High', 'Low']:
                # Create a new series to store cleaned values
                cleaned_values = pd.Series(index=df.index, dtype=float)

                for idx in df.index:
                    try:
                        # Get the raw value
                        raw_value = str(df.loc[idx, col])
                        # Remove commas
                        raw_value = raw_value.replace(',', '')

                        # Try different approaches to extract a numeric value
                        if 'Lowest:' in raw_value:
                            # Extract the number after "Lowest:"
                            value = float(raw_value.split('Lowest:')[1].strip())
                        elif raw_value.replace('.', '', 1).isdigit() or (raw_value.startswith('-') and raw_value[1:].replace('.', '', 1).isdigit()):
                            # Already a valid number
                            value = float(raw_value)
                        else:
                            # Try to extract any numeric part using regex
                            import re
                            match = re.search(r'([-+]?\d*\.?\d+)', raw_value)
                            if match:
                                value = float(match.group(1))
                            else:
                                print(f" Warning: Could not extract numeric value from '{raw_value}' in column {col}")
                                value = np.nan

                        cleaned_values.loc[idx] = value
                    except Exception as e:
                        print(f" Error processing {col} value '{df.loc[idx, col]}' at row {idx}: {str(e)}")
                        cleaned_values.loc[idx] = np.nan

                # Replace the original column with cleaned values
                df[col] = cleaned_values

                # Drop rows with NaN values in this column if necessary
                nan_count = df[col].isna().sum()
                if nan_count > 0:
                    print(f" Warning: {nan_count} rows with invalid {col} values detected")

                # Report range of values
                if df[col].notna().any():
                    print(f" {col} range: {df[col].min():.2f} to {df[col].max():.2f}")
                else:
                    print(f" Warning: No valid values found in {col}")

            # Step 2.4: Process volume (handle K/M/B suffixes)
            print(" - Processing volume column...")
            df['Volume'] = np.nan

            for idx in df.index:
                try:
                    vol_str = str(df.loc[idx, 'Vol.']).strip()

                    # Skip invalid values
                    if vol_str == '-' or vol_str.lower() == 'nan' or vol_str == '':
                        continue

                    # Convert based on suffix
                    if 'K' in vol_str:
                        # Convert thousands
                        value = float(vol_str.replace('K', '')) * 1000
                    elif 'M' in vol_str:
                        # Convert millions
                        value = float(vol_str.replace('M', '')) * 1000000
                    elif 'B' in vol_str:
                        # Convert billions
                        value = float(vol_str.replace('B', '')) * 1000000000
                    else:
                        # Direct conversion
                        value = float(vol_str)

                    df.loc[idx, 'Volume'] = value
                except Exception as e:
                    print(f" Warning: Could not convert volume '{vol_str}': {str(e)}")

            # Step 2.5: Add date-related columns
            print(" - Adding date-related columns...")
            df['Year'] = df['Date'].dt.year
            df['Quarter'] = df['Date'].dt.quarter
            df['Month'] = df['Date'].dt.month
            df['Week'] = df['Date'].dt.isocalendar().week
            df['Day'] = df['Date'].dt.day
            df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0 = Monday, 6 = Sunday

            # Step 2.6: Sort by date (earliest first)
            print(" - Sorting data by date...")
            df = df.sort_values('Date')

            # Add company name as a column
            df['Company'] = company_name

            # Store the preprocessed data
            self.preprocessed_data[company_name] = df

            # Now it's safe to visualize the preprocessed data
            self._visualize_preprocessed_data(company_name, df)

            # Report any remaining missing values
            missing_values = df.isnull().sum()
            if missing_values.sum() > 0:
                print(" - Missing values after preprocessing:")
                print(missing_values[missing_values > 0])
            else:
                print(" - No missing values in preprocessed data.")

        return self.preprocessed_data

    def _visualize_preprocessed_data(self, company_name, df):
        """Visualize preprocessed data - emphasis on visualization"""
        plt.figure(figsize=(12, 6))
        plt.plot(df['Date'], df['Price'], marker='', linestyle='-', label='Price')
        plt.title(f'Preprocessed Price Data for {company_name}')
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.legend()
        plt.tight_layout()
        plt.show()

    def fill_trading_gaps(self, method='previous'):
        """
        Fill gaps in trading days using a specified method
        'previous' (forward fill) is appropriate

        Args:
            method (str): Method to use for filling gaps
        """
        print("\n" + "="*80)
        print(f"STEP 3: FILLING TRADING GAPS USING {method.upper()}")
        print("="*80)

        filled_data = {}

        for company_name, df in self.preprocessed_data.items():
            print(f"\nFilling gaps for {company_name}...")

            # Get date range
            start_date = df['Date'].min()
            end_date = df['Date'].max()

            # Create a continuous date range
            all_dates = pd.date_range(start=start_date, end=end_date, freq='D')

            if method == 'none':
                # Just return the original data
                filled_data[company_name] = df.copy()
                print(" - Not filling gaps as method 'none' specified")
                continue

            # Create a new DataFrame with continuous date range
            continuous_df = pd.DataFrame({'Date': all_dates})

            # Merge with original data (left join to keep all dates)
            continuous_df = continuous_df.merge(df, on='Date', how='left')

            # Store original data for visualization
            original_df = df.copy()

            # Fill gaps
            if method == 'previous':
                print(" - Using forward fill method (carrying forward last available values)")

                # Forward fill numeric columns
                numeric_cols = ['Price', 'Open', 'High', 'Low', 'Volume']
                for col in numeric_cols:
                    if col in continuous_df.columns:
                        continuous_df[col] = continuous_df[col].ffill()

                # Forward fill categorical columns
                categorical_cols = ['Company', 'Year', 'Quarter', 'Month', 'Week', 'YearQuarter']
                for col in categorical_cols:
                    if col in continuous_df.columns:
                        continuous_df[col] = continuous_df[col].ffill()

                # Calculate day of week for new dates
                continuous_df['DayOfWeek'] = continuous_df['Date'].dt.dayofweek
                if 'Day' in continuous_df.columns:
                    continuous_df['Day'] = continuous_df['Date'].dt.day

                # Mark filled rows
                continuous_df['IsFilledGap'] = continuous_df['Vol.'].isna()

            elif method == 'linear':
                print(" - Using linear interpolation method")

                # Use linear interpolation for numeric columns
                numeric_cols = ['Price', 'Open', 'High', 'Low', 'Volume']
                for col in numeric_cols:
                    if col in continuous_df.columns:
                        continuous_df[col] = continuous_df[col].interpolate(method='linear')

                # Forward fill categorical columns
                categorical_cols = ['Company', 'Year', 'Quarter', 'Month', 'Week', 'YearQuarter']
                for col in categorical_cols:
                    if col in continuous_df.columns:
                        continuous_df[col] = continuous_df[col].ffill()

                # Calculate day of week for new dates
                continuous_df['DayOfWeek'] = continuous_df['Date'].dt.dayofweek
                if 'Day' in continuous_df.columns:
                    continuous_df['Day'] = continuous_df['Date'].dt.day

                # Mark filled rows
                continuous_df['IsFilledGap'] = continuous_df['Vol.'].isna()

            # Count filled gaps
            filled_gaps = continuous_df['IsFilledGap'].sum()
            print(f" - Original trading days: {len(df)}")
            print(f" - Continuous time series days: {len(continuous_df)}")
            print(f" - Gaps filled: {filled_gaps}")

            # Visualize original vs filled data
            plt.figure(figsize=(14, 6))
            plt.plot(original_df['Date'], original_df['Price'], 'bo-', label='Original Data')
            plt.plot(continuous_df['Date'], continuous_df['Price'], 'r.-', alpha=0.7, label='Filled Data')

            # Highlight filled gaps
            filled_mask = continuous_df['IsFilledGap']
            if filled_mask.sum() > 0:
                plt.scatter(continuous_df.loc[filled_mask, 'Date'],
                           continuous_df.loc[filled_mask, 'Price'],
                           color='green', alpha=0.5, s=30, label='Filled Gaps')

            plt.title(f'Original vs Filled Data for {company_name}')
            plt.xlabel('Date')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            # Store filled data
            filled_data[company_name] = continuous_df

        # Update preprocessed data with filled data
        self.preprocessed_data = filled_data

        return filled_data

    def detect_and_handle_outliers(self, method='zscore', window=20, threshold=3.0, handling='winsorize'):
        """
        Detect and handle outliers in price and volume data

        Args:
            method (str): Method for outlier detection ('zscore' or 'iqr')
            window (int): Number of previous days to use as reference
            threshold (float): Threshold for outlier detection
            handling (str): Method for handling outliers ('winsorize', 'remove', 'none')
        """
        print("\n" + "="*80)
        print(f"STEP 4: DETECTING AND HANDLING OUTLIERS")
        print("="*80)
        print(f"Using {method.upper()} method with {window}-day window and threshold of {threshold}")
        print(f"Handling method: {handling.upper()}")

        processed_data = {}

        for company_name, df in self.preprocessed_data.items():
            print(f"\nProcessing outliers for {company_name}...")

            # Create a copy for outlier detection
            outlier_df = df.copy()

            # Columns to check for outliers
            cols_to_check = ['Price', 'Volume', 'High', 'Low']

            # Create masks for each column
            outlier_masks = {}

            for col in cols_to_check:
                if col not in outlier_df.columns:
                    continue

                if method == 'zscore':
                    # Calculate Z-score based on historical data (rolling window)
                    # This avoids flagging trend movements as outliers
                    outlier_mask = np.zeros(len(outlier_df), dtype=bool)

                    # Iterate through the time series
                    # Start after having enough data points
                    for i in range(window, len(outlier_df)):
                        # Calculate mean and std using historical window
                        historical_values = outlier_df[col].iloc[max(0, i-window):i]
                        mean_until_now = historical_values.mean()
                        std_until_now = historical_values.std()

                        # Calculate z-score for current point
                        if std_until_now > 0:  # Avoid division by zero
                            z_score = (outlier_df[col].iloc[i] - mean_until_now) / std_until_now
                            if abs(z_score) > threshold:
                                outlier_mask[i] = True

                    outlier_masks[col] = outlier_mask

                elif method == 'iqr':
                    # Calculate IQRs on rolling windows
                    outlier_mask = np.zeros(len(outlier_df), dtype=bool)

                    for i in range(window, len(outlier_df)):
                        # Calculate IQR using historical window
                        historical_values = outlier_df[col].iloc[max(0, i-window):i]
                        q1 = historical_values.quantile(0.25)
                        q3 = historical_values.quantile(0.75)
                        iqr = q3 - q1

                        # Calculate bounds
                        lower_bound = q1 - threshold * iqr
                        upper_bound = q3 + threshold * iqr

                        # Check if current value is an outlier
                        if (outlier_df[col].iloc[i] < lower_bound) or (outlier_df[col].iloc[i] > upper_bound):
                            outlier_mask[i] = True

                    outlier_masks[col] = outlier_mask

                # Count outliers
                outlier_count = outlier_masks[col].sum()
                outlier_percent = (outlier_count / len(outlier_df)) * 100
                print(f" - {col} outliers: {outlier_count} ({outlier_percent:.2f}%)")

                # Visualize the outliers
                plt.figure(figsize=(14, 6))
                plt.plot(outlier_df['Date'], outlier_df[col], 'b-', label=f'{col} Values')

                # Mark outliers
                outlier_points = outlier_df[outlier_masks[col]]
                if len(outlier_points) > 0:
                    plt.scatter(outlier_points['Date'], outlier_points[col],
                               color='red', s=50, label='Detected Outliers')

                plt.title(f'{company_name} - {col} Outliers')
                plt.xlabel('Date')
                plt.ylabel(col)
                plt.legend()
                plt.grid(True, alpha=0.3)
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()

            # Mark outliers in the dataframe
            for col, mask in outlier_masks.items():
                outlier_df[f'{col}_IsOutlier'] = mask

            # Combine all outlier masks
            all_outliers = pd.Series(False, index=outlier_df.index)
            for col, mask in outlier_masks.items():
                all_outliers = all_outliers | mask

            outlier_df['IsOutlier'] = all_outliers
            total_outliers = all_outliers.sum()
            print(f" - Total records with outliers: {total_outliers} ({(total_outliers / len(outlier_df)) * 100:.2f}%)")

            # Handle outliers based on the selected method
            if handling == 'none':
                print(f" - No outlier handling applied")
                processed_df = outlier_df

            elif handling == 'winsorize':
                print(f" - Applying winsorization to outliers")
                processed_df = outlier_df.copy()

                for col in cols_to_check:
                    if col not in processed_df.columns or f'{col}_IsOutlier' not in processed_df.columns:
                        continue

                    # Get outlier mask
                    outlier_mask = processed_df[f'{col}_IsOutlier']
                    outlier_count = outlier_mask.sum()

                    if outlier_count == 0:
                        continue

                    # Store original values for visualization
                    original_values = processed_df[col].copy()

                    # Winsorize: cap values at threshold * std from mean
                    for i in range(len(processed_df)):
                        if outlier_mask[i]:
                            # Use historical data to calculate bounds
                            historical_values = processed_df[col].iloc[max(0, i-window):i]
                            if len(historical_values) > 0:
                                mean = historical_values.mean()
                                std = historical_values.std()

                                lower_bound = mean - threshold * std
                                upper_bound = mean + threshold * std

                                # Apply bounds
                                if processed_df.loc[processed_df.index[i], col] < lower_bound:
                                    processed_df.loc[processed_df.index[i], col] = lower_bound
                                elif processed_df.loc[processed_df.index[i], col] > upper_bound:
                                    processed_df.loc[processed_df.index[i], col] = upper_bound

                    print(f" - Winsorized {outlier_count} outliers in {col}")

            elif handling == 'remove':
                print(f" - Removing records with outliers")
                rows_before = len(outlier_df)
                processed_df = outlier_df[~outlier_df['IsOutlier']]
                rows_removed = rows_before - len(processed_df)
                print(f" - Removed {rows_removed} rows with outliers")

            # Visualize the effect of outlier handling
            if handling != 'none' and 'Price' in cols_to_check:
                plt.figure(figsize=(14, 6))
                plt.plot(outlier_df['Date'], outlier_df['Price'], 'b-', alpha=0.5, label='Before Handling')
                plt.plot(processed_df['Date'], processed_df['Price'], 'r-', label='After Handling')

                # Mark original outliers
                outlier_points = outlier_df[outlier_df['Price_IsOutlier']]
                if len(outlier_points) > 0:
                    plt.scatter(outlier_points['Date'], outlier_points['Price'],
                               color='red', s=50, marker='x', label='Detected Outliers')

                plt.title(f'{company_name} - Price After Outlier Handling ({handling})')
                plt.xlabel('Date')
                plt.ylabel('Price')
                plt.legend()
                plt.grid(True, alpha=0.3)
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()

            processed_data[company_name] = processed_df

        # Update preprocessed data
        self.preprocessed_data = processed_data

        return processed_data

    #################################
    # SECTION 3: FEATURE ENGINEERING
    #################################
    def engineer_features(self):
        """
        Create enhanced technical indicators and features for stock prediction
        Including:
        - Bollinger Bands with dynamic parameters
        - Enhanced RSI with multiple timeframes
        - Market regime detection
        - Multi-signal indicator system
        """
        print("\n" + "="*80)
        print("STEP 5: FEATURE ENGINEERING")
        print("="*80)

        featured_data = {}

        for company_name, df in self.preprocessed_data.items():
            print(f"\nCreating features for {company_name}...")

            # Create a copy to work with
            df_featured = df.copy()

            # Get company-specific parameters or use defaults
            bollinger_window = self.market_params.get(company_name, {}).get('bollinger_window', 20)

            # Step 5.1: Calculate Simple Moving Averages (SMA)
            print(" - Calculating Simple Moving Averages (SMA)...")
            df_featured['SMA_5'] = df_featured['Price'].rolling(window=5).mean()
            df_featured['SMA_7'] = df_featured['Price'].rolling(window=7).mean()
            df_featured['SMA_10'] = df_featured['Price'].rolling(window=10).mean()
            df_featured['SMA_20'] = df_featured['Price'].rolling(window=20).mean()  # Key for Bollinger Bands

            # Exponential Moving Averages for additional signals
            print(" - Calculating Exponential Moving Averages (EMA)...")
            df_featured['EMA_5'] = df_featured['Price'].ewm(span=5, adjust=False).mean()
            df_featured['EMA_7'] = df_featured['Price'].ewm(span=7, adjust=False).mean()
            df_featured['EMA_12'] = df_featured['Price'].ewm(span=12, adjust=False).mean()
            df_featured['EMA_26'] = df_featured['Price'].ewm(span=26, adjust=False).mean()

            # Visualize moving averages
            plt.figure(figsize=(14, 6))
            plt.plot(df_featured['Date'], df_featured['Price'], label='Price', color='black')
            plt.plot(df_featured['Date'], df_featured['SMA_5'], label='SMA 5', alpha=0.7)
            plt.plot(df_featured['Date'], df_featured['SMA_7'], label='SMA 7', alpha=0.7)
            plt.plot(df_featured['Date'], df_featured['SMA_20'], label='SMA 20', alpha=0.7)
            plt.title(f'{company_name} - Price with Moving Averages')
            plt.xlabel('Date')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            # Step 5.2: Calculate Bollinger Bands
            print(f" - Calculating Bollinger Bands with {bollinger_window}-day window...")
            # Standard 20-day Bollinger Bands (or using market-specific window)
            df_featured[f'BB_middle_{bollinger_window}'] = df_featured['Price'].rolling(window=bollinger_window).mean()
            df_featured[f'BB_std_{bollinger_window}'] = df_featured['Price'].rolling(window=bollinger_window).std()
            df_featured[f'BB_upper_{bollinger_window}'] = df_featured[f'BB_middle_{bollinger_window}'] + (2 * df_featured[f'BB_std_{bollinger_window}'])
            df_featured[f'BB_lower_{bollinger_window}'] = df_featured[f'BB_middle_{bollinger_window}'] - (2 * df_featured[f'BB_std_{bollinger_window}'])

            # For compatibility with original code
            df_featured['BB_middle'] = df_featured[f'BB_middle_{bollinger_window}']
            df_featured['BB_std'] = df_featured[f'BB_std_{bollinger_window}']
            df_featured['BB_upper'] = df_featured[f'BB_upper_{bollinger_window}']
            df_featured['BB_lower'] = df_featured[f'BB_lower_{bollinger_window}']

            # Calculate position within bands (0 to 1, where 0 is at lower band, 1 is at upper band)
            df_featured['BB_position'] = (df_featured['Price'] - df_featured['BB_lower']) / (df_featured['BB_upper'] - df_featured['BB_lower'])

            # Calculate Bollinger Band width - useful for volatility assessment
            df_featured['BB_width'] = (df_featured['BB_upper'] - df_featured['BB_lower']) / df_featured['BB_middle']

            # Visualize Bollinger Bands
            plt.figure(figsize=(14, 6))
            plt.plot(df_featured['Date'], df_featured['Price'], label='Price', color='blue')
            plt.plot(df_featured['Date'], df_featured['BB_upper'], label=f'Upper Band ({bollinger_window}-day)', color='red', linestyle='--')
            plt.plot(df_featured['Date'], df_featured['BB_middle'], label=f'Middle Band ({bollinger_window}-day)', color='black')
            plt.plot(df_featured['Date'], df_featured['BB_lower'], label=f'Lower Band ({bollinger_window}-day)', color='green', linestyle='--')
            plt.title(f'{company_name} - Bollinger Bands (Using {bollinger_window}-day Window)')
            plt.xlabel('Date')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            # Step 5.3: Calculate Price Momentum Indicators
            print(" - Calculating Price Momentum Indicators...")
            # Price changes over different periods
            df_featured['Price_1d_change'] = df_featured['Price'].pct_change(1)
            df_featured['Price_5d_change'] = df_featured['Price'].pct_change(5)
            df_featured['Price_7d_change'] = df_featured['Price'].pct_change(7)

            # Rate of change (ROC)
            df_featured['ROC_5'] = ((df_featured['Price'] - df_featured['Price'].shift(5)) / df_featured['Price'].shift(5)) * 100
            df_featured['ROC_7'] = ((df_featured['Price'] - df_featured['Price'].shift(7)) / df_featured['Price'].shift(7)) * 100

            # Visualize price momentum
            plt.figure(figsize=(14, 6))
            plt.plot(df_featured['Date'], df_featured['Price_1d_change'], label='1-day Change')
            plt.plot(df_featured['Date'], df_featured['Price_5d_change'], label='5-day Change')
            plt.plot(df_featured['Date'], df_featured['Price_7d_change'], label='7-day Change')
            plt.title(f'{company_name} - Price Momentum (% Change)')
            plt.xlabel('Date')
            plt.ylabel('Percent Change')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            # Step 5.4: Calculate Relative Strength Index (RSI)
            print(" - Calculating RSI (Relative Strength Index)...")
            # Calculate price differences
            delta = df_featured['Price'].diff()

            # Create masks for gains and losses
            gain = delta.where(delta > 0, 0)
            loss = -delta.where(delta < 0, 0)

            # Calculate average gains and losses over different periods
            # 7-day RSI
            avg_gain_7 = gain.rolling(window=7).mean()
            avg_loss_7 = loss.rolling(window=7).mean()
            rs_7 = avg_gain_7 / avg_loss_7
            df_featured['RSI_7'] = 100 - (100 / (1 + rs_7))

            # 14-day RSI (standard)
            avg_gain_14 = gain.rolling(window=14).mean()
            avg_loss_14 = loss.rolling(window=14).mean()
            rs_14 = avg_gain_14 / avg_loss_14
            df_featured['RSI_14'] = 100 - (100 / (1 + rs_14))

            # Visualize RSI
            plt.figure(figsize=(14, 6))
            plt.plot(df_featured['Date'], df_featured['RSI_7'], label='RSI 7-day')
            plt.plot(df_featured['Date'], df_featured['RSI_14'], label='RSI 14-day')
            plt.axhline(y=70, color='r', linestyle='--', alpha=0.5)  # Overbought
            plt.axhline(y=30, color='g', linestyle='--', alpha=0.5)  # Oversold
            plt.title(f'{company_name} - Relative Strength Index')
            plt.xlabel('Date')
            plt.ylabel('RSI')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            # Step 5.5: Calculate MACD (Moving Average Convergence Divergence)
            print(" - Calculating MACD...")
            df_featured['MACD'] = df_featured['EMA_12'] - df_featured['EMA_26']
            df_featured['MACD_signal'] = df_featured['MACD'].ewm(span=9, adjust=False).mean()
            df_featured['MACD_histogram'] = df_featured['MACD'] - df_featured['MACD_signal']

            # Visualize MACD
            plt.figure(figsize=(14, 6))
            plt.plot(df_featured['Date'], df_featured['MACD'], label='MACD', color='blue')
            plt.plot(df_featured['Date'], df_featured['MACD_signal'], label='Signal', color='red')
            plt.bar(df_featured['Date'], df_featured['MACD_histogram'], label='Histogram', color='green', alpha=0.5)
            plt.axhline(y=0, color='black', linestyle='-', alpha=0.2)
            plt.title(f'{company_name} - MACD')
            plt.xlabel('Date')
            plt.ylabel('Value')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            # Step 5.6: Volume-Based Features (Important for addressing underprediction issues)
            print(" - Calculating Volume-Based Features...")
            df_featured['Volume_1d_change'] = df_featured['Volume'].pct_change(1)
            df_featured['Volume_5d_MA'] = df_featured['Volume'].rolling(window=5).mean()
            df_featured['Volume_10d_MA'] = df_featured['Volume'].rolling(window=10).mean()
            df_featured['Volume_20d_MA'] = df_featured['Volume'].rolling(window=20).mean()  # Added 20-day MA

            # Volume ratio (current volume to recent average) - highlights unusual volume
            df_featured['Volume_ratio_5d'] = df_featured['Volume'] / df_featured['Volume_5d_MA']
            df_featured['Volume_ratio_10d'] = df_featured['Volume'] / df_featured['Volume_10d_MA']
            df_featured['Volume_ratio_20d'] = df_featured['Volume'] / df_featured['Volume_20d_MA']  # Added 20-day ratio

            # For compatibility with original code
            df_featured['Volume_ratio'] = df_featured['Volume_ratio_10d']

            # Relationship between price and volume changes
            df_featured['Price_Volume_change'] = df_featured['Price_1d_change'] * df_featured['Volume_1d_change']

            # Price-Volume trend indicator (positive when price and volume move in same direction)
            df_featured['Price_Volume_trend'] = np.sign(df_featured['Price_1d_change']) * np.sign(df_featured['Volume_1d_change'])

            # On-Balance Volume (OBV)
            df_featured['OBV'] = 0
            for i in range(1, len(df_featured)):
                if df_featured['Price'].iloc[i] > df_featured['Price'].iloc[i-1]:
                    df_featured.loc[df_featured.index[i], 'OBV'] = df_featured['OBV'].iloc[i-1] + df_featured['Volume'].iloc[i]
                elif df_featured['Price'].iloc[i] < df_featured['Price'].iloc[i-1]:
                    df_featured.loc[df_featured.index[i], 'OBV'] = df_featured['OBV'].iloc[i-1] - df_featured['Volume'].iloc[i]
                else:
                    df_featured.loc[df_featured.index[i], 'OBV'] = df_featured['OBV'].iloc[i-1]

            # Visualize Volume Features
            plt.figure(figsize=(14, 10))

            # Plot 1: Volume and Moving Average
            plt.subplot(3, 1, 1)
            plt.bar(df_featured['Date'], df_featured['Volume'], alpha=0.5)
            plt.plot(df_featured['Date'], df_featured['Volume_5d_MA'], color='red', label='5-day MA')
            plt.plot(df_featured['Date'], df_featured['Volume_20d_MA'], color='blue', label='20-day MA')  # Added 20-day MA
            plt.title(f'{company_name} - Volume and Moving Averages')
            plt.ylabel('Volume')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 2: Volume Ratio
            plt.subplot(3, 1, 2)
            plt.plot(df_featured['Date'], df_featured['Volume_ratio_5d'], label='5-day Ratio')
            plt.plot(df_featured['Date'], df_featured['Volume_ratio_20d'], label='20-day Ratio')  # Added 20-day ratio
            plt.axhline(y=1, color='r', linestyle='--', alpha=0.3)
            plt.title(f'{company_name} - Volume Ratio (Volume / MA)')
            plt.ylabel('Ratio')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 3: OBV
            plt.subplot(3, 1, 3)
            plt.plot(df_featured['Date'], df_featured['OBV'])
            plt.title(f'{company_name} - On-Balance Volume (OBV)')
            plt.xlabel('Date')
            plt.ylabel('OBV')
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

            # Step 5.7: Volatility Indicators
            print(" - Calculating Volatility Indicators...")
            # Calculate True Range
            tr1 = df_featured['High'] - df_featured['Low']
            tr2 = abs(df_featured['High'] - df_featured['Price'].shift(1))
            tr3 = abs(df_featured['Low'] - df_featured['Price'].shift(1))
            df_featured['TR'] = pd.DataFrame({'tr1': tr1, 'tr2': tr2, 'tr3': tr3}).max(axis=1)

            # Average True Range (ATR)
            df_featured['ATR_7'] = df_featured['TR'].rolling(window=7).mean()
            df_featured['ATR_14'] = df_featured['TR'].rolling(window=14).mean()
            df_featured['ATR_20'] = df_featured['TR'].rolling(window=20).mean()  # Added 20-day ATR

            # Historical Volatility
            df_featured['Volatility_5d'] = df_featured['Price_1d_change'].rolling(window=5).std() * np.sqrt(5)
            df_featured['Volatility_7d'] = df_featured['Price_1d_change'].rolling(window=7).std() * np.sqrt(7)
            df_featured['Volatility_20d'] = df_featured['Price_1d_change'].rolling(window=20).std() * np.sqrt(20)  # Added 20-day volatility

            # Visualize Volatility
            plt.figure(figsize=(14, 6))
            plt.plot(df_featured['Date'], df_featured['ATR_7'], label='ATR 7-day')
            plt.plot(df_featured['Date'], df_featured['ATR_20'], label='ATR 20-day')  # Added 20-day ATR
            plt.plot(df_featured['Date'], df_featured['Volatility_7d'], label='7-day Volatility', alpha=0.7)
            plt.plot(df_featured['Date'], df_featured['Volatility_20d'], label='20-day Volatility', alpha=0.7)  # Added 20-day volatility
            plt.title(f'{company_name} - Volatility Indicators')
            plt.xlabel('Date')
            plt.ylabel('Value')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            # Step 5.8: Market Regime Detection (ADX)
            print(" - Detecting Market Regimes using ADX...")
            # Calculate Directional Movement
            df_featured['DM_plus'] = 0.0
            df_featured['DM_minus'] = 0.0

            # Calculate DM+ and DM-
            for i in range(1, len(df_featured)):
                up_move = df_featured['High'].iloc[i] - df_featured['High'].iloc[i-1]
                down_move = df_featured['Low'].iloc[i-1] - df_featured['Low'].iloc[i]

                if up_move > down_move and up_move > 0:
                    df_featured.loc[df_featured.index[i], 'DM_plus'] = up_move
                else:
                    df_featured.loc[df_featured.index[i], 'DM_plus'] = 0.0

                if down_move > up_move and down_move > 0:
                    df_featured.loc[df_featured.index[i], 'DM_minus'] = down_move
                else:
                    df_featured.loc[df_featured.index[i], 'DM_minus'] = 0.0

            # Smooth DM values with 14-period average
            period = 14
            df_featured['DM_plus_14'] = df_featured['DM_plus'].rolling(window=period).sum()
            df_featured['DM_minus_14'] = df_featured['DM_minus'].rolling(window=period).sum()

            # Calculate Directional Index (DI+ and DI-)
            df_featured['DI_plus'] = 100 * (df_featured['DM_plus_14'] / df_featured['ATR_14'])
            df_featured['DI_minus'] = 100 * (df_featured['DM_minus_14'] / df_featured['ATR_14'])

            # Calculate Directional Movement Index (DX)
            df_featured['DX'] = 100 * abs(df_featured['DI_plus'] - df_featured['DI_minus']) / (df_featured['DI_plus'] + df_featured['DI_minus'])

            # Calculate Average Directional Index (ADX)
            df_featured['ADX'] = df_featured['DX'].rolling(window=period).mean()

            # Determine Market Regime
            df_featured['Market_Regime'] = 'UNKNOWN'
            df_featured.loc[df_featured['ADX'] > 25, 'Market_Regime'] = 'TRENDING'
            df_featured.loc[df_featured['ADX'] <= 25, 'Market_Regime'] = 'RANGING'

            # Calculate dynamic parameters based on market regime
            df_featured['Dynamic_StopLoss'] = df_featured['Volatility_7d'] * 1.5  # 1.5x current volatility
            df_featured['Dynamic_TakeProfit'] = df_featured['Volatility_7d'] * 2.5  # 2.5x current volatility

            # Adjust holding period based on regime
            df_featured['Max_Hold_Period'] = 10  # Default
            df_featured.loc[df_featured['Market_Regime'] == 'TRENDING', 'Max_Hold_Period'] = 15  # Longer for trending
            df_featured.loc[df_featured['Market_Regime'] == 'RANGING', 'Max_Hold_Period'] = 5  # Shorter for ranging

            # Visualize Market Regime
            plt.figure(figsize=(14, 10))

            # Plot 1: Price and ADX
            plt.subplot(3, 1, 1)
            plt.plot(df_featured['Date'], df_featured['Price'], label='Price', color='blue')
            plt.title(f'{company_name} - Price Chart')
            plt.ylabel('Price')
            plt.grid(True, alpha=0.3)
            plt.legend()

            # Plot 2: ADX
            plt.subplot(3, 1, 2)
            plt.plot(df_featured['Date'], df_featured['ADX'], label='ADX', color='purple')
            plt.axhline(y=25, color='r', linestyle='--', alpha=0.5)
            plt.title('Average Directional Index (ADX)')
            plt.ylabel('ADX')
            plt.grid(True, alpha=0.3)
            plt.legend()

            # Plot 3: Market Regime
            plt.subplot(3, 1, 3)
            regime_numeric = df_featured['Market_Regime'].map({'TRENDING': 1, 'RANGING': 0, 'UNKNOWN': np.nan})
            plt.plot(df_featured['Date'], regime_numeric, 'g-')
            plt.yticks([0, 1], ['RANGING', 'TRENDING'])
            plt.title('Market Regime')
            plt.xlabel('Date')
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

            # Step 5.9: Generate Bollinger Bands Trading Signals
            print(" - Generating Bollinger Bands Trading Signals...")
            df_featured['BB_Signal'] = 'HOLD'

            # Simple Bollinger Bands Strategy
            # Buy when price touches or crosses below the lower band
            buy_condition = (df_featured['Price'] <= df_featured['BB_lower'])
            df_featured.loc[buy_condition, 'BB_Signal'] = 'BUY'

            # Sell when price touches or crosses above the upper band
            sell_condition = (df_featured['Price'] >= df_featured['BB_upper'])
            df_featured.loc[sell_condition, 'BB_Signal'] = 'SELL'

            # Generate enhanced signals with additional indicators
            df_featured['Enhanced_Signal'] = df_featured['BB_Signal']  # Start with BB signals

            # Enhanced buy conditions: BB + RSI + Volume
            enhanced_buy = (
                (df_featured['Price'] <= df_featured['BB_lower']) &  # Price at/below lower BB
                (df_featured['RSI_7'] < 40) &                        # Not too overbought
                (df_featured['Volume'] > df_featured['Volume_5d_MA'])  # Higher than avg volume
            )
            df_featured.loc[enhanced_buy, 'Enhanced_Signal'] = 'BUY'

            # Enhanced sell conditions: BB + RSI + Volume
            enhanced_sell = (
                (df_featured['Price'] >= df_featured['BB_upper']) &  # Price at/above upper BB
                (df_featured['RSI_7'] > 60) &                        # Not too oversold
                (df_featured['Volume'] > df_featured['Volume_5d_MA'])  # Higher than avg volume
            )
            df_featured.loc[enhanced_sell, 'Enhanced_Signal'] = 'SELL'

            # Create buy/sell scores as a measure of signal strength
            df_featured['Buy_Score'] = 0
            df_featured['Sell_Score'] = 0

            # Bollinger Band signals
            df_featured.loc[df_featured['Price'] < df_featured['BB_lower'], 'Buy_Score'] += 2
            df_featured.loc[df_featured['Price'] > df_featured['BB_upper'], 'Sell_Score'] += 2

            # RSI signals
            df_featured.loc[df_featured['RSI_7'] < 30, 'Buy_Score'] += 1.5
            df_featured.loc[df_featured['RSI_7'] > 70, 'Sell_Score'] += 1.5

            # MACD signals
            df_featured.loc[df_featured['MACD_histogram'] > 0, 'Buy_Score'] += 1
            df_featured.loc[df_featured['MACD_histogram'] < 0, 'Sell_Score'] += 1

            # Volume confirmation
            df_featured.loc[df_featured['Volume'] > df_featured['Volume_5d_MA'] * 1.5, 'Buy_Score'] += 1
            df_featured.loc[df_featured['Volume'] > df_featured['Volume_5d_MA'] * 1.5, 'Sell_Score'] += 1

            # Visualize Bollinger Bands Trading Signals
            plt.figure(figsize=(14, 8))

            # Plot 1: Price with Bollinger Bands and Signals
            plt.subplot(2, 1, 1)
            plt.plot(df_featured['Date'], df_featured['Price'], label='Price', color='blue')
            plt.plot(df_featured['Date'], df_featured['BB_upper'], label='Upper Band', color='red', linestyle='--')
            plt.plot(df_featured['Date'], df_featured['BB_lower'], label='Lower Band', color='green', linestyle='--')

            # Mark BUY and SELL signals from Bollinger Bands
            buy_signals = df_featured[df_featured['BB_Signal'] == 'BUY']
            sell_signals = df_featured[df_featured['BB_Signal'] == 'SELL']

            if not buy_signals.empty:
                plt.scatter(buy_signals['Date'], buy_signals['Price'], marker='^', s=100,
                           color='green', label='BB Buy Signal')

            if not sell_signals.empty:
                plt.scatter(sell_signals['Date'], sell_signals['Price'], marker='v', s=100,
                           color='red', label='BB Sell Signal')

            plt.title(f'{company_name} - Bollinger Bands Trading Signals')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 2: Enhanced Signals and Signal Strength
            plt.subplot(2, 1, 2)
            plt.plot(df_featured['Date'], df_featured['Price'], label='Price', color='blue')

            # Mark enhanced BUY and SELL signals
            enhanced_buy_signals = df_featured[df_featured['Enhanced_Signal'] == 'BUY']
            enhanced_sell_signals = df_featured[df_featured['Enhanced_Signal'] == 'SELL']

            if not enhanced_buy_signals.empty:
                plt.scatter(enhanced_buy_signals['Date'], enhanced_buy_signals['Price'],
                           marker='^', s=100, color='green', label='Enhanced Buy')

            if not enhanced_sell_signals.empty:
                plt.scatter(enhanced_sell_signals['Date'], enhanced_sell_signals['Price'],
                           marker='v', s=100, color='red', label='Enhanced Sell')

            # Show buy/sell scores as bars
            plt.bar(df_featured['Date'], df_featured['Buy_Score'] * 0.05,
                   label='Buy Score', color='green', alpha=0.3)
            plt.bar(df_featured['Date'], df_featured['Sell_Score'] * -0.05,
                   label='Sell Score', color='red', alpha=0.3)

            plt.title(f'{company_name} - Enhanced Trading Signals')
            plt.xlabel('Date')
            plt.ylabel('Price / Signal Score')
            plt.legend()
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

            # Step 5.10: Calculate target variables for prediction
            print(" - Creating Target Variables for Prediction...")
            # Next day's price (regression target)
            df_featured['Next_Day_Price'] = df_featured['Price'].shift(-1)

            # Price change next day (for regression)
            df_featured['Price_Change_Next_Day'] = df_featured['Next_Day_Price'] / df_featured['Price'] - 1

            # Binary target: Will price go up tomorrow? (for classification)
            df_featured['Price_Up_Next_Day'] = (df_featured['Price_Change_Next_Day'] > 0).astype(int)

            # Drop rows with NaN values (due to shifts and rolling windows)
            nan_count_before = df_featured.isnull().sum().sum()
            df_featured = df_featured.dropna()
            nan_count_after = df_featured.isnull().sum().sum()
            print(f" - Dropped {len(df) - len(df_featured)} rows with NaN values")
            print(f" - Final shape of featured data: {df_featured.shape}")

            # Store the engineered data
            featured_data[company_name] = df_featured

            # Analyze feature correlations with target
            self._analyze_feature_correlations(company_name, df_featured)

        self.featured_data = featured_data
        return featured_data

    def _analyze_feature_correlations(self, company_name, df):
        """Analyze feature correlations with the target variable"""
        print(f"\nFeature Correlation Analysis for {company_name}:")

        # Select only numeric columns
        numeric_df = df.select_dtypes(include=[np.number])

        # Calculate correlation with target variables
        corr_with_target = numeric_df.corr()['Price_Change_Next_Day'].sort_values(ascending=False)

        print("\nTop 10 Features Positively Correlated with Next Day's Price Change:")
        print(corr_with_target.head(10))

        print("\nTop 10 Features Negatively Correlated with Next Day's Price Change:")
        print(corr_with_target.tail(10))

        # Plot correlation heatmap for key features
        plt.figure(figsize=(14, 12))

        # Select key features for visualization
        key_features = [
            'Price', 'Volume', 'SMA_7', 'RSI_7', 'MACD',
            'Volume_ratio', 'ATR_7', 'BB_position',
            'Next_Day_Price', 'Price_Change_Next_Day'
        ]

        # Make sure all features exist
        key_features = [f for f in key_features if f in numeric_df.columns]

        # Create correlation matrix
        corr_matrix = numeric_df[key_features].corr()

        # Plot heatmap
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
        plt.title(f'Correlation Matrix of Key Features for {company_name}')
        plt.tight_layout()
        plt.show()

    #################################
    # SECTION 4: DATA SPLITTING AND MODEL TRAINING
    #################################
    def split_data(self, test_year=2021, test_quarter=1):
        """
        Split data into training and testing sets
        Following project requirements: 2020 data for training, 2021 Q1 for testing
        """
        print("\n" + "="*80)
        print("STEP 6: SPLITTING DATA INTO TRAIN AND TEST SETS")
        print("="*80)

        split_datasets = {}

        for company_name, df in self.featured_data.items():
            print(f"\nSplitting data for {company_name}...")

            # Define training set (data before test_year)
            train_data = df[df['Year'] < test_year]

            # Define testing set (data from test_year, test_quarter)
            test_data = df[(df['Year'] == test_year) & (df['Quarter'] == test_quarter)]

            print(f" - Training set shape: {train_data.shape}")
            print(f" - Testing set shape: {test_data.shape}")

            # Visualize the train/test split
            plt.figure(figsize=(14, 6))
            plt.plot(train_data['Date'], train_data['Price'], label=f'Training Data ({test_year-1})', color='blue')
            plt.plot(test_data['Date'], test_data['Price'], label=f'Testing Data ({test_year} Q{test_quarter})', color='red')
            plt.axvline(x=pd.to_datetime(f"{test_year}-01-01"), color='black', linestyle='--')
            plt.title(f'{company_name} - Train/Test Split')
            plt.xlabel('Date')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            # Store split datasets
            split_datasets[company_name] = {
                'train': train_data,
                'test': test_data
            }

        return split_datasets

    def create_sequences(self, data, features, target='Next_Day_Price', seq_length=None):
        """
        Create sequences for sequence models
        Using configurable sequence length (5-7 days recommended)

        Args:
            data (DataFrame): Input data
            features (list): List of feature columns to use
            target (str): Target column name
            seq_length (int): Sequence length (window size)

        Returns:
            tuple: X sequences and y target values
        """
        if seq_length is None:
            seq_length = self.params['seq_length']

        X, y = [], []

        for i in range(len(data) - seq_length):
            # Get sequence of features
            X.append(data[features].iloc[i:(i+seq_length)].values)
            # Get target value
            y.append(data[target].iloc[i+seq_length])

        return np.array(X), np.array(y)

    #################################
    # SECTION 5: MODEL ARCHITECTURES
    #################################
    class LSTMModel(nn.Module):
        """
        LSTM model for time series prediction
        Implementing attention mechanism for better feature focus
        """
        def __init__(self, input_dim, hidden_dim=64, num_layers=3, dropout=0.2):
            super().__init__()
            self.hidden_dim = hidden_dim
            self.num_layers = num_layers

            # LSTM layers with dropout
            self.lstm = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0
            )

            # Attention mechanism
            self.attention = nn.Linear(hidden_dim, 1)

            # Fully connected output layer
            self.fc = nn.Linear(hidden_dim, 1)

        def forward(self, x):
            # Forward propagate LSTM
            # x shape: [batch_size, seq_length, input_dim]
            lstm_out, _ = self.lstm(x)  # lstm_out: [batch_size, seq_length, hidden_dim]

            # Apply attention
            attention_weights = F.softmax(self.attention(lstm_out), dim=1)
            context_vector = torch.sum(attention_weights * lstm_out, dim=1)

            # Final prediction
            out = self.fc(context_vector)

            return out

    class TransformerModel(nn.Module):
        """
        Transformer model for time series prediction
        Enhanced with positional encoding and multi-head attention
        """
        def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, dropout=0.1):
            super().__init__()

            # Embedding layer to convert input features to model dimension
            self.embedding = nn.Linear(input_dim, d_model)

            # Create positional encoding
            self.pos_encoding = self.create_positional_encoding(d_model, max_len=100)

            # Transformer encoder layer
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=d_model*4,
                dropout=dropout,
                batch_first=True
            )

            # Transformer encoder
            self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

            # Output layer
            self.output_layer = nn.Linear(d_model, 1)

            # Dropout
            self.dropout = nn.Dropout(dropout)

        def create_positional_encoding(self, d_model, max_len=100):
            """Create positional encoding for transformer model"""
            position = torch.arange(max_len).unsqueeze(1).float()
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

            pe = torch.zeros(max_len, d_model)
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)

            # Add batch dimension [1, max_len, d_model]
            pe = pe.unsqueeze(0)

            # Register as buffer (won't be considered as model parameters)
            return pe

        def forward(self, x):
            # x shape: [batch_size, seq_length, input_dim]

            # Embed features
            x = self.embedding(x)  # [batch_size, seq_length, d_model]

            # Add positional encoding
            x = x + self.pos_encoding[:, :x.size(1), :].to(x.device)
            x = self.dropout(x)

            # Pass through transformer encoder
            x = self.transformer_encoder(x)  # [batch_size, seq_length, d_model]

            # Take the last time step output
            x = x[:, -1, :]  # [batch_size, d_model]

            # Output layer
            x = self.output_layer(x)  # [batch_size, 1]

            return x

    #################################
    # SECTION 6: MODEL TRAINING
    #################################
    def train_lstm_model(self, company_name, split_data, features=None, target='Next_Day_Price',
                          seq_length=None, params=None):
        """
        Train an LSTM model for time series prediction
        Incorporating:
        - Configurable sequence length (5-7 days)
        - Attention mechanism for better feature focus
        - 3-layer architecture for better learning

        Args:
            company_name (str): Name of the company
            split_data (dict): Dictionary with train and test data
            features (list): List of features to use (if None, use default set)
            target (str): Target variable to predict
            seq_length (int): Sequence length for LSTM (if None, use default)
            params (dict): Model parameters (if None, use default)

        Returns:
            dict: Model information including the trained model and results
        """
        print("\n" + "="*80)
        print(f"STEP 7: TRAINING LSTM MODEL FOR {company_name}")
        print("="*80)

        # Use default parameters if not specified
        if params is None:
            params = self.params

        # Use company-specific parameters if available
        if company_name in self.market_params:
            company_params = self.market_params[company_name]
            if 'seq_length' in company_params and seq_length is None:
                seq_length = company_params['seq_length']

        if seq_length is None:
            seq_length = params['seq_length']

        # Use default features if not specified
        if features is None:
            features = [
                'Price', 'Open', 'High', 'Low',  # Price components
                'Volume', 'Volume_ratio',        # Volume features (mentor emphasized)
                'SMA_7', 'RSI_7',                # Using 7-day indicators
                'MACD', 'ATR_7', 'BB_position'   # Additional features
            ]

        print(f"Training LSTM model with sequence length of {seq_length} days")
        print(f"Using features: {', '.join(features)}")

        # Get train and test data
        train_data = split_data['train']
        test_data = split_data['test']

        # Create sequences for LSTM
        X_train, y_train = self.create_sequences(train_data, features, target, seq_length)
        X_test, y_test = self.create_sequences(test_data, features, target, seq_length)

        print(f"Training sequences shape: {X_train.shape}")
        print(f"Testing sequences shape: {X_test.shape}")

        # Scale the data
        scaler_X = MinMaxScaler()
        scaler_y = MinMaxScaler()

        # Reshape X for scaling (to 2D)
        X_train_2d = X_train.reshape(-1, X_train.shape[-1])
        X_test_2d = X_test.reshape(-1, X_test.shape[-1])

        # Fit and transform
        X_train_2d = scaler_X.fit_transform(X_train_2d)
        X_test_2d = scaler_X.transform(X_test_2d)

        # Reshape back to 3D
        X_train = X_train_2d.reshape(X_train.shape)
        X_test = X_test_2d.reshape(X_test.shape)

        # Scale y
        y_train = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_test = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

        # Convert to PyTorch tensors
        X_train = torch.FloatTensor(X_train)
        y_train = torch.FloatTensor(y_train)
        X_test = torch.FloatTensor(X_test)
        y_test = torch.FloatTensor(y_test)

        # Create dataset and dataloader
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)

        # Initialize the LSTM model
        model = self.LSTMModel(
            input_dim=len(features),
            hidden_dim=params['hidden_dim'],
            num_layers=params['num_layers'],
            dropout=params['dropout']
        )

        # Define loss function and optimizer
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])

        # Training loop
        print(f"Training LSTM model for {params['num_epochs']} epochs...")
        loss_history = []

        for epoch in range(params['num_epochs']):
            model.train()
            total_loss = 0

            for X_batch, y_batch in train_loader:
                # Forward pass
                outputs = model(X_batch)
                loss = criterion(outputs.flatten(), y_batch)

                # Backward pass and optimize
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            # Calculate average loss for this epoch
            avg_loss = total_loss / len(train_loader)
            loss_history.append(avg_loss)

            # Print progress every 10 epochs
            if (epoch + 1) % 10 == 0:
                print(f" Epoch {epoch+1}/{params['num_epochs']}, Loss: {avg_loss:.6f}")

        print("LSTM training complete!")

        # Evaluate the model on test data
        model.eval()
        with torch.no_grad():
            test_predictions = model(X_test).numpy().flatten()

            # Inverse transform to original scale
            test_predictions = scaler_y.inverse_transform(test_predictions.reshape(-1, 1)).flatten()
            test_actuals = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

            # Calculate error metrics
            mae = mean_absolute_error(test_actuals, test_predictions)
            mse = mean_squared_error(test_actuals, test_predictions)
            rmse = np.sqrt(mse)
            r2 = r2_score(test_actuals, test_predictions)

            # Calculate MAPE - preferred metric for time series (emphasized by mentor)
            mape = np.mean(np.abs((test_actuals - test_predictions) / test_actuals)) * 100

            # Calculate directional accuracy - secondary metric mentioned by mentor
            actual_direction = np.sign(np.diff(np.append([test_actuals[0]], test_actuals)))
            pred_direction = np.sign(np.diff(np.append([test_predictions[0]], test_predictions)))
            directional_accuracy = np.mean(actual_direction == pred_direction) * 100

        # Print evaluation metrics
        print("\nEvaluation Metrics on Test Data:")
        print(f" MAPE: {mape:.2f}% (Mean Absolute Percentage Error)")
        print(f" MAE: {mae:.4f} (Mean Absolute Error)")
        print(f" RMSE: {rmse:.4f} (Root Mean Squared Error)")
        print(f" R²: {r2:.4f} (Coefficient of Determination)")
        print(f" Directional Accuracy: {directional_accuracy:.2f}%")

        # Prepare dates for plotting
        test_dates = test_data['Date'][seq_length:len(test_actuals) + seq_length].reset_index(drop=True)

        # Create DataFrame with results for visualization (emphasized by mentor)
        results_df = pd.DataFrame({
            'Date': test_dates,
            'Actual': test_actuals,
            'Predicted': test_predictions
        })

        # Visualize predictions vs actual values
        plt.figure(figsize=(14, 10))

        # Plot 1: Actual vs Predicted
        plt.subplot(2, 1, 1)
        plt.plot(results_df['Date'], results_df['Actual'], 'b-', label='Actual')
        plt.plot(results_df['Date'], results_df['Predicted'], 'r--', label='Predicted')
        plt.title(f'{company_name} - LSTM Predictions vs Actual Prices')
        plt.ylabel('Price')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Plot 2: Errors
        plt.subplot(2, 1, 2)
        plt.bar(results_df['Date'], results_df['Actual'] - results_df['Predicted'], color='g', alpha=0.7)
        plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
        plt.title('Prediction Errors (Actual - Predicted)')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Analyze prediction patterns - emphasis on understanding "why"
        # visual analysis of where predictions fail
        self._analyze_prediction_patterns(company_name, test_actuals, test_predictions, test_dates)

        # Store model and results
        model_info = {
            'model': model,
            'scaler_X': scaler_X,
            'scaler_y': scaler_y,
            'features': features,
            'target': target,
            'seq_length': seq_length,
            'params': params,
            'loss_history': loss_history,
            'results_df': results_df,
            'metrics': {
                'mape': mape,
                'mae': mae,
                'rmse': rmse,
                'r2': r2,
                'directional_accuracy': directional_accuracy
            }
        }

        return model_info

    def train_transformer_model(self, company_name, split_data, features=None, target='Next_Day_Price',
                                seq_length=None, params=None):
        """
        Train a Transformer model for time series prediction
        Enhanced with positional encoding and multi-head attention

        Args:
            company_name (str): Name of the company
            split_data (dict): Dictionary with train and test data
            features (list): List of features to use (if None, use default set)
            target (str): Target variable to predict
            seq_length (int): Sequence length for Transformer (if None, use default)
            params (dict): Model parameters (if None, use default)

        Returns:
            dict: Model information including the trained model and results
        """
        print("\n" + "="*80)
        print(f"STEP 8: TRAINING TRANSFORMER MODEL FOR {company_name}")
        print("="*80)

        # Use default parameters if not specified
        if params is None:
            params = {
                'seq_length': self.params['seq_length'],
                'd_model': 64,  # Embedding dimension
                'nhead': 4,     # Number of attention heads
                'num_layers': 2, # Number of transformer layers
                'dropout': 0.1,
                'batch_size': self.params['batch_size'],
                'num_epochs': self.params['num_epochs'],
                'learning_rate': self.params['learning_rate']
            }

        # Use company-specific parameters if available
        if company_name in self.market_params:
            company_params = self.market_params[company_name]
            if 'seq_length' in company_params and seq_length is None:
                seq_length = company_params['seq_length']

        if seq_length is None:
            seq_length = params['seq_length']

        # Use default features if not specified - same as LSTM
        if features is None:
            features = [
                'Price', 'Open', 'High', 'Low',
                'Volume', 'Volume_ratio',
                'SMA_7', 'RSI_7',
                'MACD', 'ATR_7', 'BB_position'
            ]

        print(f"Training Transformer model with sequence length of {seq_length} days")
        print(f"Using features: {', '.join(features)}")

        # Get train and test data
        train_data = split_data['train']
        test_data = split_data['test']

        # Create sequences
        X_train, y_train = self.create_sequences(train_data, features, target, seq_length)
        X_test, y_test = self.create_sequences(test_data, features, target, seq_length)

        print(f"Training sequences shape: {X_train.shape}")
        print(f"Testing sequences shape: {X_test.shape}")

        # Scale the data
        scaler_X = MinMaxScaler()
        scaler_y = MinMaxScaler()

        # Reshape X for scaling (to 2D)
        X_train_2d = X_train.reshape(-1, X_train.shape[-1])
        X_test_2d = X_test.reshape(-1, X_test.shape[-1])

        # Fit and transform
        X_train_2d = scaler_X.fit_transform(X_train_2d)
        X_test_2d = scaler_X.transform(X_test_2d)

        # Reshape back to 3D
        X_train = X_train_2d.reshape(X_train.shape)
        X_test = X_test_2d.reshape(X_test.shape)

        # Scale y
        y_train = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_test = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

        # Convert to PyTorch tensors
        X_train = torch.FloatTensor(X_train)
        y_train = torch.FloatTensor(y_train)
        X_test = torch.FloatTensor(X_test)
        y_test = torch.FloatTensor(y_test)

        # Create dataset and dataloader
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)

        # Initialize the Transformer model
        model = self.TransformerModel(
            input_dim=len(features),
            d_model=params['d_model'],
            nhead=params['nhead'],
            num_layers=params['num_layers'],
            dropout=params['dropout']
        )

        # Define loss function and optimizer
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])

        # Training loop
        print(f"Training Transformer model for {params['num_epochs']} epochs...")
        loss_history = []

        for epoch in range(params['num_epochs']):
            model.train()
            total_loss = 0

            for X_batch, y_batch in train_loader:
                # Forward pass
                outputs = model(X_batch)
                loss = criterion(outputs.flatten(), y_batch)

                # Backward pass and optimize
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            # Calculate average loss for this epoch
            avg_loss = total_loss / len(train_loader)
            loss_history.append(avg_loss)

            # Print progress every 10 epochs
            if (epoch + 1) % 10 == 0:
                print(f" Epoch {epoch+1}/{params['num_epochs']}, Loss: {avg_loss:.6f}")

        print("Transformer training complete!")

        # Evaluate the model on test data - same evaluation as LSTM
        model.eval()
        with torch.no_grad():
            test_predictions = model(X_test).numpy().flatten()

            # Inverse transform to original scale
            test_predictions = scaler_y.inverse_transform(test_predictions.reshape(-1, 1)).flatten()
            test_actuals = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

            # Calculate error metrics
            mae = mean_absolute_error(test_actuals, test_predictions)
            mse = mean_squared_error(test_actuals, test_predictions)
            rmse = np.sqrt(mse)
            r2 = r2_score(test_actuals, test_predictions)

            # Calculate MAPE
            mape = np.mean(np.abs((test_actuals - test_predictions) / test_actuals)) * 100

            # Calculate directional accuracy
            actual_direction = np.sign(np.diff(np.append([test_actuals[0]], test_actuals)))
            pred_direction = np.sign(np.diff(np.append([test_predictions[0]], test_predictions)))
            directional_accuracy = np.mean(actual_direction == pred_direction) * 100

        # Print evaluation metrics
        print("\nEvaluation Metrics on Test Data:")
        print(f" MAPE: {mape:.2f}% (Mean Absolute Percentage Error)")
        print(f" MAE: {mae:.4f} (Mean Absolute Error)")
        print(f" RMSE: {rmse:.4f} (Root Mean Squared Error)")
        print(f" R²: {r2:.4f} (Coefficient of Determination)")
        print(f" Directional Accuracy: {directional_accuracy:.2f}%")

        # Prepare dates for plotting
        test_dates = test_data['Date'][seq_length:len(test_actuals) + seq_length].reset_index(drop=True)

        # Create DataFrame with results
        results_df = pd.DataFrame({
            'Date': test_dates,
            'Actual': test_actuals,
            'Predicted': test_predictions
        })

        # Visualize predictions vs actual values - same visualization as LSTM
        plt.figure(figsize=(14, 10))

        # Plot 1: Actual vs Predicted
        plt.subplot(2, 1, 1)
        plt.plot(results_df['Date'], results_df['Actual'], 'b-', label='Actual')
        plt.plot(results_df['Date'], results_df['Predicted'], 'r--', label='Predicted')
        plt.title(f'{company_name} - Transformer Predictions vs Actual Prices')
        plt.ylabel('Price')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Plot 2: Errors
        plt.subplot(2, 1, 2)
        plt.bar(results_df['Date'], results_df['Actual'] - results_df['Predicted'], color='g', alpha=0.7)
        plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
        plt.title('Prediction Errors (Actual - Predicted)')
        plt.xlabel('Date')
        plt.ylabel('Error')
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Analyze prediction patterns - same analysis as LSTM
        self._analyze_prediction_patterns(company_name, test_actuals, test_predictions, test_dates)

        # Store model and results
        model_info = {
            'model': model,
            'scaler_X': scaler_X,
            'scaler_y': scaler_y,
            'features': features,
            'target': target,
            'seq_length': seq_length,
            'params': params,
            'loss_history': loss_history,
            'results_df': results_df,
            'metrics': {
                'mape': mape,
                'mae': mae,
                'rmse': rmse,
                'r2': r2,
                'directional_accuracy': directional_accuracy
            }
        }

        return model_info

    def _analyze_prediction_patterns(self, company_name, actuals, predictions, dates):
        """
        Analyze prediction patterns to understand model behavior
        Following emphasis on understanding "why" models behave as they do

        """
        # Calculate errors
        errors = actuals - predictions
        percent_errors = (errors / actuals) * 100

        # Create DataFrame for analysis
        analysis_df = pd.DataFrame({
            'Date': dates,
            'Actual': actuals,
            'Predicted': predictions,
            'Error': errors,
            'AbsError': np.abs(errors),
            'PctError': percent_errors
        })

        # Add day of week
        analysis_df['DayOfWeek'] = analysis_df['Date'].dt.dayofweek
        analysis_df['WeekdayName'] = analysis_df['Date'].dt.day_name()

        # Analyze errors by price range (quantile)
        price_bins = pd.qcut(analysis_df['Actual'], 4, labels=['Low', 'Medium-Low', 'Medium-High', 'High'])
        analysis_df['PriceRange'] = price_bins

        # Examine error patterns
        print("\nAnalyzing Prediction Patterns:")

        # Systematic bias
        bias = np.mean(errors)
        bias_pct = (bias / np.mean(actuals)) * 100
        print(f" Systematic Bias: {bias:.4f} ({bias_pct:.2f}%)")

        if bias < 0:
            print(" The model tends to OVERPREDICT prices (predictions higher than actuals)")
        else:
            print(" The model tends to UNDERPREDICT prices (predictions lower than actuals)")

        # Error patterns by price range
        print("\n Error Analysis by Price Range:")
        price_range_analysis = analysis_df.groupby('PriceRange')['PctError'].agg(['mean', 'std', 'count'])
        print(price_range_analysis)

        # Error patterns by day of week
        print("\n Error Analysis by Day of Week:")
        weekday_analysis = analysis_df.groupby('WeekdayName')['PctError'].agg(['mean', 'std', 'count'])
        print(weekday_analysis)

        # Visualize error patterns
        plt.figure(figsize=(14, 10))

        # Plot 1: Error by price range
        plt.subplot(2, 1, 1)
        sns.boxplot(x='PriceRange', y='PctError', data=analysis_df)
        plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
        plt.title(f'{company_name} - Prediction Errors by Price Range')
        plt.ylabel('Percentage Error (%)')
        plt.grid(True, alpha=0.3)

        # Plot 2: Error by day of week
        plt.subplot(2, 1, 2)
        sns.boxplot(x='WeekdayName', y='PctError', data=analysis_df)
        plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
        plt.title('Prediction Errors by Day of Week')
        plt.ylabel('Percentage Error (%)')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    #################################
    # SECTION 7: MODEL COMPARISON
    #################################
    def compare_models(self, company_name, models_results):
        """
        Compare different models for the same company
        Enhanced with focus on directional accuracy and trading performance

        Args:
            company_name (str): Name of the company
            models_results (dict): Dictionary with results from different models

        Returns:
            dict: Comparison results
        """
        print("\n" + "="*80)
        print(f"STEP 9: COMPARING MODELS FOR {company_name}")
        print("="*80)

        # Extract metrics for each model
        metrics_data = []

        for model_name, results in models_results.items():
            if 'metrics' in results:
                metrics = results['metrics']
                metrics_data.append({
                    'Model': model_name,
                    'MAPE (%)': metrics.get('mape', float('nan')),
                    'MAE': metrics.get('mae', float('nan')),
                    'RMSE': metrics.get('rmse', float('nan')),
                    'R²': metrics.get('r2', float('nan')),
                    'Directional Accuracy (%)': metrics.get('directional_accuracy', float('nan'))
                })

        # Create DataFrame for comparison
        comparison_df = pd.DataFrame(metrics_data)

        # Print comparison table
        print("\nModel Comparison Metrics:")
        print(comparison_df.to_string(index=False))

        # Find the best model based on MAPE
        if len(comparison_df) > 0:
            best_model_idx = comparison_df['MAPE (%)'].idxmin()
            best_model = comparison_df.iloc[best_model_idx]['Model']
            print(f"\nBest model for {company_name}: {best_model}")
            print(f" MAPE: {comparison_df.iloc[best_model_idx]['MAPE (%)']:.2f}%")
            print(f" Directional Accuracy: {comparison_df.iloc[best_model_idx]['Directional Accuracy (%)']:.2f}%")
        else:
            print("No models to compare.")
            return None

        # Visualize comparison
        plt.figure(figsize=(14, 10))

        # Plot 1: MAPE Comparison (lower is better)
        plt.subplot(2, 1, 1)
        colors = ['blue'] * len(comparison_df)
        if len(comparison_df) > 0:
            colors[best_model_idx] = 'green'

        plt.bar(comparison_df['Model'], comparison_df['MAPE (%)'], color=colors)
        plt.title(f'{company_name} - Model Comparison: MAPE (lower is better)')
        plt.ylabel('MAPE (%)')
        plt.grid(True, alpha=0.3, axis='y')

        # Plot 2: Directional Accuracy Comparison (higher is better)
        plt.subplot(2, 1, 2)
        plt.bar(comparison_df['Model'], comparison_df['Directional Accuracy (%)'], color=colors)
        plt.title(f'{company_name} - Model Comparison: Directional Accuracy (higher is better)')
        plt.ylabel('Directional Accuracy (%)')
        plt.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.show()

        # Compare predictions from different models
        plt.figure(figsize=(14, 7))

        # Get actual values
        dates = models_results[list(models_results.keys())[0]]['results_df']['Date']
        actuals = models_results[list(models_results.keys())[0]]['results_df']['Actual']

        # Plot actual values
        plt.plot(dates, actuals, 'k-', label='Actual', linewidth=2)

        # Plot predictions from each model
        for model_name, results in models_results.items():
            if 'results_df' in results:
                predictions = results['results_df']['Predicted']
                plt.plot(dates, predictions, '--', label=f'{model_name} Predicted', alpha=0.7)

        plt.title(f'{company_name} - Model Predictions Comparison')
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        return {
            'company': company_name,
            'comparison_df': comparison_df,
            'best_model': best_model,
            'best_metrics': comparison_df.iloc[best_model_idx].to_dict()
        }

    #################################
    # SECTION 8: TRADING STRATEGIES
    #################################
    def implement_bollinger_bands_strategy(self, company_name, data, initial_capital=10000, visualize=True):
        """
        Implement a Bollinger Bands trading strategy.
        This is a baseline strategy to compare with more advanced approaches

        Args:
            company_name (str): Name of the company
            data (DataFrame): Data containing price and Bollinger Bands
            initial_capital (float): Initial investment amount
            visualize (bool): Whether to visualize the results

        Returns:
            dict: Trading strategy results
        """
        print("\n" + "="*80)
        print(f"STEP 10: IMPLEMENTING BOLLINGER BANDS STRATEGY FOR {company_name}")
        print("="*80)
        print(f"Initial capital: ${initial_capital:.2f}")

        # Get company-specific parameters
        bollinger_window = self.market_params.get(company_name, {}).get('bollinger_window', 20)

        # Create a copy of the dataframe
        df = data.copy()

        # Make sure we have Bollinger Bands calculated with the appropriate window
        if not all(col in df.columns for col in [f'BB_upper_{bollinger_window}', f'BB_middle_{bollinger_window}', f'BB_lower_{bollinger_window}']):
            print(f"Calculating Bollinger Bands ({bollinger_window}-day, 2 std)...")
            df[f'BB_middle_{bollinger_window}'] = df['Price'].rolling(window=bollinger_window).mean()
            df[f'BB_std_{bollinger_window}'] = df['Price'].rolling(window=bollinger_window).std()
            df[f'BB_upper_{bollinger_window}'] = df[f'BB_middle_{bollinger_window}'] + (2 * df[f'BB_std_{bollinger_window}'])
            df[f'BB_lower_{bollinger_window}'] = df[f'BB_middle_{bollinger_window}'] - (2 * df[f'BB_std_{bollinger_window}'])

            # For compatibility
            df['BB_middle'] = df[f'BB_middle_{bollinger_window}']
            df['BB_upper'] = df[f'BB_upper_{bollinger_window}']
            df['BB_lower'] = df[f'BB_lower_{bollinger_window}']

        # Generate trading signals based on Bollinger Bands
        df['BB_Signal'] = 'HOLD'

        # Buy when price crosses below the lower band
        buy_condition = (df['Price'] <= df['BB_lower'])
        df.loc[buy_condition, 'BB_Signal'] = 'BUY'

        # Sell when price crosses above the upper band
        sell_condition = (df['Price'] >= df['BB_upper'])
        df.loc[sell_condition, 'BB_Signal'] = 'SELL'

        # Print signals summary
        signal_counts = df['BB_Signal'].value_counts()
        print("\nBollinger Bands trading signals:")
        for signal, count in signal_counts.items():
            print(f" {signal}: {count}")

        # Initialize portfolio tracking
        df['Position'] = 0  # Number of shares held
        df['Capital'] = initial_capital  # Available cash
        df['Holdings'] = 0  # Value of holdings
        df['Portfolio'] = initial_capital  # Total portfolio value
        df['Action'] = ""  # Action taken

        # Trading strategy simulation
        current_position = 0
        entry_price = 0

        for i in range(1, len(df)):
            current_date = df.iloc[i]['Date']
            current_price = df.iloc[i]['Price']
            current_signal = df.iloc[i]['BB_Signal']

            if current_position > 0:
                # We have a position already
                if current_signal == 'SELL':
                    # Sell based on Bollinger signal
                    capital = df.iloc[i-1]['Capital'] + current_position * current_price
                    action = f"SELL {current_position} shares at ${current_price:.2f}"
                    current_position = 0
                    entry_price = 0
                else:
                    # Continue holding
                    capital = df.iloc[i-1]['Capital']
                    action = "HOLD"
            else:
                # No position
                if current_signal == 'BUY':
                    # Calculate how many shares we can buy
                    available_capital = df.iloc[i-1]['Capital']
                    max_shares = int(available_capital / current_price)

                    if max_shares > 0:
                        # Enter position
                        current_position = max_shares
                        entry_price = current_price
                        capital = available_capital - (current_position * current_price)
                        action = f"BUY {current_position} shares at ${current_price:.2f}"
                    else:
                        action = "HOLD (insufficient capital)"
                        capital = available_capital
                else:
                    # No action needed
                    action = "HOLD"
                    capital = df.iloc[i-1]['Capital']

            # Update tracking columns
            df.iloc[i, df.columns.get_loc('Position')] = current_position
            df.iloc[i, df.columns.get_loc('Capital')] = capital
            df.iloc[i, df.columns.get_loc('Holdings')] = current_position * current_price
            df.iloc[i, df.columns.get_loc('Portfolio')] = capital + (current_position * current_price)
            df.iloc[i, df.columns.get_loc('Action')] = action

        # Calculate strategy performance
        initial_value = initial_capital
        final_value = df.iloc[-1]['Portfolio']
        total_return = (final_value - initial_value) / initial_value * 100

        # Calculate buy and hold return for comparison
        first_price = df.iloc[0]['Price']
        last_price = df.iloc[-1]['Price']
        buy_hold_shares = initial_capital / first_price
        buy_hold_value = buy_hold_shares * last_price
        buy_hold_return = (buy_hold_value - initial_capital) / initial_capital * 100

        # Calculate trading metrics
        buy_actions = [action for action in df['Action'] if 'BUY' in action]
        sell_actions = [action for action in df['Action'] if 'SELL' in action]

        # Print strategy performance
        print("\nBollinger Bands Strategy Performance:")
        print(f" Initial Capital: ${initial_value:.2f}")
        print(f" Final Portfolio Value: ${final_value:.2f}")
        print(f" Total Return: {total_return:.2f}%")
        print(f" Buy & Hold Return: {buy_hold_return:.2f}%")
        print(f" Outperformance: {total_return - buy_hold_return:.2f}%")
        print(f" Number of Trades: {len(buy_actions)}")

        if visualize:
            # Define RSI thresholds for visualization
            rsi_threshold_high = 70
            rsi_threshold_low = 30

            # Visualize trading strategy
            plt.figure(figsize=(14, 15))

            # Plot 1: Price with Bollinger Bands
            plt.subplot(3, 1, 1)
            plt.plot(df['Date'], df['Price'], label='Price', color='blue')
            plt.plot(df['Date'], df['BB_upper'], label=f'Upper Band ({bollinger_window}-day)', color='red', linestyle='--')
            plt.plot(df['Date'], df['BB_middle'], label=f'Middle Band ({bollinger_window}-day)', color='black')
            plt.plot(df['Date'], df['BB_lower'], label=f'Lower Band ({bollinger_window}-day)', color='green', linestyle='--')

            # Mark buy and sell points
            buy_points = df[df['Action'].str.contains('BUY')]
            sell_points = df[df['Action'].str.contains('SELL')]

            if len(buy_points) > 0:
                plt.scatter(buy_points['Date'], buy_points['Price'], marker='^', s=100, color='green', label='Buy Point')

            if len(sell_points) > 0:
                plt.scatter(sell_points['Date'], sell_points['Price'], marker='v', s=100, color='red', label='Sell Point')

            plt.title(f'{company_name} - Bollinger Bands Strategy (Using {bollinger_window}-day Window)')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 2: RSI
            plt.subplot(3, 1, 2)
            plt.plot(df['Date'], df['RSI_7'], label='RSI (7-day)', color='purple')
            plt.axhline(y=rsi_threshold_high, color='r', linestyle='--', alpha=0.5, label=f'Overbought ({rsi_threshold_high})')
            plt.axhline(y=rsi_threshold_low, color='g', linestyle='--', alpha=0.5, label=f'Oversold ({rsi_threshold_low})')
            plt.title(f'{company_name} - RSI Confirmation')
            plt.ylabel('RSI')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 3: Portfolio Value
            plt.subplot(3, 1, 3)
            plt.plot(df['Date'], df['Portfolio'], label='BB Strategy', color='blue')

            # Add buy & hold line for comparison
            buy_hold_value_series = initial_capital * (df['Price'] / df['Price'].iloc[0])
            plt.plot(df['Date'], buy_hold_value_series, label='Buy & Hold', color='green', linestyle='--')

            plt.title(f'{company_name} - Portfolio Value')
            plt.xlabel('Date')
            plt.ylabel('Value ($)')
            plt.legend()
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

        # Store results
        strategy_results = {
            'company': company_name,
            'strategy_type': 'Bollinger Bands',
            'strategy_df': df,
            'initial_capital': initial_capital,
            'final_value': final_value,
            'total_return': total_return,
            'buy_hold_return': buy_hold_return,
            'outperformance': total_return - buy_hold_return,
            'trades_count': len(buy_actions)
        }

        return strategy_results

    def implement_enhanced_bollinger_bands_strategy(self, company_name, data, initial_capital=10000, visualize=True):
        """
        Implement an enhanced Bollinger Bands strategy with RSI and volume filters

        Args:
            company_name (str): Name of the company
            data (DataFrame): Data containing price and technical indicators
            initial_capital (float): Initial investment amount
            visualize (bool): Whether to visualize the results

        Returns:
            dict: Trading strategy results
        """
        print("\n" + "="*80)
        print(f"STEP 11: IMPLEMENTING ENHANCED BOLLINGER BANDS STRATEGY FOR {company_name}")
        print("="*80)
        print(f"Initial capital: ${initial_capital:.2f}")

        # Get company-specific parameters
        bollinger_window = self.market_params.get(company_name, {}).get('bollinger_window', 20)

        # Create a copy of the dataframe
        df = data.copy()

        # Define RSI thresholds
        rsi_threshold_high = 70
        rsi_threshold_low = 30

        # Generate enhanced trading signals
        df['Enhanced_Signal'] = 'HOLD'

        # Enhanced buy condition: Price at lower BB + RSI oversold + Volume confirmation
        buy_condition = (
            (df['Price'] <= df['BB_lower']) &  # Price at/below lower BB
            (df['RSI_7'] < rsi_threshold_low) &  # RSI oversold
            (df['Volume'] > df['Volume_5d_MA'])  # Higher than average volume
        )
        df.loc[buy_condition, 'Enhanced_Signal'] = 'BUY'

        # Enhanced sell condition: Price at upper BB + RSI overbought + Volume confirmation
        sell_condition = (
            (df['Price'] >= df['BB_upper']) &  # Price at/above upper BB
            (df['RSI_7'] > rsi_threshold_high) &  # RSI overbought
            (df['Volume'] > df['Volume_5d_MA'])  # Higher than average volume
        )
        df.loc[sell_condition, 'Enhanced_Signal'] = 'SELL'

        # Print signals summary
        signal_counts = df['Enhanced_Signal'].value_counts()
        print("\nEnhanced strategy trading signals:")
        for signal, count in signal_counts.items():
            print(f" {signal}: {count}")

        # Initialize portfolio tracking
        df['Position'] = 0  # Number of shares held
        df['Capital'] = initial_capital  # Available cash
        df['Holdings'] = 0  # Value of holdings
        df['Portfolio'] = initial_capital  # Total portfolio value
        df['Action'] = ""  # Action taken

        # Trading strategy simulation
        current_position = 0
        entry_price = 0

        for i in range(1, len(df)):
            current_date = df.iloc[i]['Date']
            current_price = df.iloc[i]['Price']
            current_signal = df.iloc[i]['Enhanced_Signal']

            if current_position > 0:
                # We have a position already
                if current_signal == 'SELL':
                    # Sell based on enhanced signal
                    capital = df.iloc[i-1]['Capital'] + current_position * current_price
                    action = f"SELL {current_position} shares at ${current_price:.2f}"
                    current_position = 0
                    entry_price = 0
                else:
                    # Continue holding
                    capital = df.iloc[i-1]['Capital']
                    action = "HOLD"
            else:
                # No position
                if current_signal == 'BUY':
                    # Calculate how many shares we can buy
                    available_capital = df.iloc[i-1]['Capital']
                    max_shares = int(available_capital / current_price)

                    if max_shares > 0:
                        # Enter position
                        current_position = max_shares
                        entry_price = current_price
                        capital = available_capital - (current_position * current_price)
                        action = f"BUY {current_position} shares at ${current_price:.2f}"
                    else:
                        action = "HOLD (insufficient capital)"
                        capital = available_capital
                else:
                    # No action needed
                    action = "HOLD"
                    capital = df.iloc[i-1]['Capital']

            # Update tracking columns
            df.iloc[i, df.columns.get_loc('Position')] = current_position
            df.iloc[i, df.columns.get_loc('Capital')] = capital
            df.iloc[i, df.columns.get_loc('Holdings')] = current_position * current_price
            df.iloc[i, df.columns.get_loc('Portfolio')] = capital + (current_position * current_price)
            df.iloc[i, df.columns.get_loc('Action')] = action

        # Calculate strategy performance
        initial_value = initial_capital
        final_value = df.iloc[-1]['Portfolio']
        total_return = (final_value - initial_value) / initial_value * 100

        # Calculate buy and hold return for comparison
        first_price = df.iloc[0]['Price']
        last_price = df.iloc[-1]['Price']
        buy_hold_shares = initial_capital / first_price
        buy_hold_value = buy_hold_shares * last_price
        buy_hold_return = (buy_hold_value - initial_capital) / initial_capital * 100

        # Calculate trading metrics
        buy_actions = [action for action in df['Action'] if 'BUY' in action]
        sell_actions = [action for action in df['Action'] if 'SELL' in action]

        # Print strategy performance
        print("\nEnhanced Bollinger Bands Strategy Performance:")
        print(f" Initial Capital: ${initial_value:.2f}")
        print(f" Final Portfolio Value: ${final_value:.2f}")
        print(f" Total Return: {total_return:.2f}%")
        print(f" Buy & Hold Return: {buy_hold_return:.2f}%")
        print(f" Outperformance: {total_return - buy_hold_return:.2f}%")
        print(f" Number of Trades: {len(buy_actions)}")

        if visualize:
            # Visualize trading strategy
            plt.figure(figsize=(14, 15))

            # Plot 1: Price with Bollinger Bands
            plt.subplot(3, 1, 1)
            plt.plot(df['Date'], df['Price'], label='Price', color='blue')
            plt.plot(df['Date'], df['BB_upper'], label=f'Upper Band ({bollinger_window}-day)', color='red', linestyle='--')
            plt.plot(df['Date'], df['BB_middle'], label=f'Middle Band ({bollinger_window}-day)', color='black')
            plt.plot(df['Date'], df['BB_lower'], label=f'Lower Band ({bollinger_window}-day)', color='green', linestyle='--')

            # Mark buy and sell points
            buy_points = df[df['Action'].str.contains('BUY')]
            sell_points = df[df['Action'].str.contains('SELL')]

            if len(buy_points) > 0:
                plt.scatter(buy_points['Date'], buy_points['Price'], marker='^', s=100, color='green', label='Buy Point')

            if len(sell_points) > 0:
                plt.scatter(sell_points['Date'], sell_points['Price'], marker='v', s=100, color='red', label='Sell Point')

            plt.title(f'{company_name} - Enhanced Bollinger Bands Strategy')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 2: RSI with thresholds
            plt.subplot(3, 1, 2)
            plt.plot(df['Date'], df['RSI_7'], label='RSI (7-day)', color='purple')
            plt.axhline(y=rsi_threshold_high, color='r', linestyle='--', alpha=0.5, label=f'Overbought ({rsi_threshold_high})')
            plt.axhline(y=rsi_threshold_low, color='g', linestyle='--', alpha=0.5, label=f'Oversold ({rsi_threshold_low})')
            plt.title(f'{company_name} - RSI with Trading Zones')
            plt.ylabel('RSI')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 3: Portfolio Value
            plt.subplot(3, 1, 3)
            plt.plot(df['Date'], df['Portfolio'], label='Enhanced BB Strategy', color='blue')

            # Add buy & hold line for comparison
            buy_hold_value_series = initial_capital * (df['Price'] / df['Price'].iloc[0])
            plt.plot(df['Date'], buy_hold_value_series, label='Buy & Hold', color='green', linestyle='--')

            plt.title(f'{company_name} - Portfolio Value')
            plt.xlabel('Date')
            plt.ylabel('Value ($)')
            plt.legend()
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

        # Store results
        strategy_results = {
            'company': company_name,
            'strategy_type': 'Enhanced Bollinger Bands with RSI',
            'strategy_df': df,
            'initial_capital': initial_capital,
            'final_value': final_value,
            'total_return': total_return,
            'buy_hold_return': buy_hold_return,
            'outperformance': total_return - buy_hold_return,
            'trades_count': len(buy_actions),
            'parameters': {
                'rsi_threshold_low': rsi_threshold_low,
                'rsi_threshold_high': rsi_threshold_high
            }
        }

        return strategy_results

    def implement_model_based_strategy(self, company_name, model_results, initial_capital=10000,
                                       buy_threshold=0.005, sell_threshold=-0.005,
                                       holding_period_limit=10, stop_loss_pct=0.05, take_profit_pct=0.10,
                                       visualize=True):
        """
        Implement a trading strategy based on model predictions

        Args:
            company_name (str): Name of the company
            model_results (dict): Results from a trained model
            initial_capital (float): Initial investment amount
            buy_threshold (float): Threshold for buy signals (predicted change %)
            sell_threshold (float): Threshold for sell signals (predicted change %)
            holding_period_limit (int): Maximum days to hold a position
            stop_loss_pct (float): Stop loss percentage
            take_profit_pct (float): Take profit percentage
            visualize (bool): Whether to visualize the results

        Returns:
            dict: Trading strategy results
        """
        print("\n" + "="*80)
        print(f"STEP 12: IMPLEMENTING MODEL-BASED STRATEGY FOR {company_name}")
        print("="*80)
        print(f"Initial capital: ${initial_capital:.2f}")
        print(f"Buy threshold: {buy_threshold:.4f}, Sell threshold: {sell_threshold:.4f}")
        print(f"Maximum holding period: {holding_period_limit} days")
        print(f"Stop loss: {stop_loss_pct*100:.1f}%, Take profit: {take_profit_pct*100:.1f}%")

        # Get results DataFrame with actual and predicted prices
        results_df = model_results['results_df'].copy()

        # Calculate predicted price change (percent)
        results_df['Predicted_Change_Pct'] = results_df['Predicted'].pct_change(1).shift(-1)
        results_df['Actual_Change_Pct'] = results_df['Actual'].pct_change(1).shift(-1)

        # Initialize trading signal based on predicted change
        # BUY if predicted change is positive above threshold
        # SELL if predicted change is negative below threshold
        # HOLD otherwise
        results_df['Signal'] = 'HOLD'
        results_df.loc[results_df['Predicted_Change_Pct'] > buy_threshold, 'Signal'] = 'BUY'
        results_df.loc[results_df['Predicted_Change_Pct'] < sell_threshold, 'Signal'] = 'SELL'

        # Print signals count
        signal_counts = results_df['Signal'].value_counts()
        print("\nTrading signals based on predicted price changes:")
        for signal, count in signal_counts.items():
            print(f" {signal}: {count}")

        # Initialize portfolio tracking columns
        results_df['Position'] = 0  # Number of shares held
        results_df['Capital'] = initial_capital  # Available cash
        results_df['Holdings'] = 0  # Value of holdings
        results_df['Portfolio'] = initial_capital  # Total portfolio value
        results_df['Action'] = ""  # Action taken
        results_df['Days_Held'] = 0  # Days holding current position
        results_df['Entry_Price'] = 0  # Entry price for current position

        # Trading strategy simulation
        current_position = 0
        entry_price = 0
        entry_date = None
        days_held = 0

        for i in range(len(results_df)):
            current_date = results_df.iloc[i]['Date']
            current_price = results_df.iloc[i]['Actual']
            current_signal = results_df.iloc[i]['Signal']

            # Update days held if we have a position
            if current_position > 0:
                days_held += 1
                results_df.iloc[i, results_df.columns.get_loc('Days_Held')] = days_held

                # Calculate current profit/loss
                current_return = (current_price - entry_price) / entry_price

                # Determine if we should exit the position
                if days_held >= holding_period_limit:
                    # Exit due to holding period limit (minimizing hold periods)
                    action = "SELL (hold limit)"
                    capital = results_df.iloc[i-1]['Capital'] + current_position * current_price
                    current_position = 0
                    days_held = 0
                    entry_price = 0
                elif current_return <= -stop_loss_pct:
                    # Exit due to stop loss
                    action = "SELL (stop loss)"
                    capital = results_df.iloc[i-1]['Capital'] + current_position * current_price
                    current_position = 0
                    days_held = 0
                    entry_price = 0
                elif current_return >= take_profit_pct:
                    # Exit due to take profit
                    action = "SELL (take profit)"
                    capital = results_df.iloc[i-1]['Capital'] + current_position * current_price
                    current_position = 0
                    days_held = 0
                    entry_price = 0
                elif current_signal == 'SELL':
                    # Exit due to model signal
                    action = "SELL (signal)"
                    capital = results_df.iloc[i-1]['Capital'] + current_position * current_price
                    current_position = 0
                    days_held = 0
                    entry_price = 0
                else:
                    # Continue holding
                    action = "HOLD"
                    capital = results_df.iloc[i-1]['Capital']
            else:  # No position
                if current_signal == 'BUY':
                    # Calculate how many shares we can buy with available capital
                    available_capital = results_df.iloc[i-1]['Capital']
                    max_shares = int(available_capital / current_price)

                    if max_shares > 0:
                        # Enter new position
                        current_position = max_shares
                        entry_price = current_price
                        entry_date = current_date
                        days_held = 0
                        capital = available_capital - (current_position * current_price)
                        action = f"BUY {current_position} shares at ${current_price:.2f}"
                    else:
                        action = "HOLD (insufficient capital)"
                        capital = available_capital
                else:
                    # No action
                    action = "HOLD (no signal)"
                    capital = results_df.iloc[i-1]['Capital']

            # Update tracking columns
            results_df.iloc[i, results_df.columns.get_loc('Position')] = current_position
            results_df.iloc[i, results_df.columns.get_loc('Capital')] = capital
            results_df.iloc[i, results_df.columns.get_loc('Holdings')] = current_position * current_price
            results_df.iloc[i, results_df.columns.get_loc('Portfolio')] = capital + (current_position * current_price)
            results_df.iloc[i, results_df.columns.get_loc('Action')] = action
            results_df.iloc[i, results_df.columns.get_loc('Entry_Price')] = entry_price

        # Calculate strategy performance
        initial_value = initial_capital
        final_value = results_df.iloc[-1]['Portfolio']
        total_return = (final_value - initial_value) / initial_value * 100

        # Calculate buy and hold return
        first_price = results_df.iloc[0]['Actual']
        last_price = results_df.iloc[-1]['Actual']
        buy_hold_shares = initial_capital / first_price
        buy_hold_value = buy_hold_shares * last_price
        buy_hold_return = (buy_hold_value - initial_capital) / initial_capital * 100

        # Calculate trading metrics
        buy_actions = [action for action in results_df['Action'] if 'BUY' in action]
        sell_actions = [action for action in results_df['Action'] if 'SELL' in action]

        # Calculate average holding period
        non_zero_holding_periods = results_df[results_df['Days_Held'] > 0]['Days_Held']
        avg_holding_period = non_zero_holding_periods.mean() if len(non_zero_holding_periods) > 0 else 0

        # Print strategy performance
        print("\nModel-Based Trading Strategy Performance:")
        print(f" Initial Capital: ${initial_value:.2f}")
        print(f" Final Portfolio Value: ${final_value:.2f}")
        print(f" Total Return: {total_return:.2f}%")
        print(f" Buy & Hold Return: {buy_hold_return:.2f}%")
        print(f" Outperformance: {total_return - buy_hold_return:.2f}%")
        print(f" Number of Trades: {len(buy_actions)}")
        print(f" Average Holding Period: {avg_holding_period:.2f} days")

        if visualize:
            # Visualize trading strategy performance
            plt.figure(figsize=(14, 12))

            # Plot 1: Portfolio Value
            plt.subplot(3, 1, 1)
            plt.plot(results_df['Date'], results_df['Portfolio'], label='Model Strategy', color='blue')

            # Calculate buy and hold portfolio value over time
            results_df['BuyHold_Value'] = initial_capital * results_df['Actual'] / results_df['Actual'].iloc[0]
            plt.plot(results_df['Date'], results_df['BuyHold_Value'], label='Buy & Hold', color='green', linestyle='--')

            plt.title(f'{company_name} - Model-Based Trading Strategy Performance')
            plt.ylabel('Portfolio Value ($)')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 2: Stock Price with Buy/Sell Actions
            plt.subplot(3, 1, 2)
            plt.plot(results_df['Date'], results_df['Actual'], label='Stock Price', color='black')

            # Mark buy and sell points
            buy_points = results_df[results_df['Action'].str.contains('BUY')]
            sell_points = results_df[results_df['Action'].str.contains('SELL')]

            if len(buy_points) > 0:
                plt.scatter(buy_points['Date'], buy_points['Actual'], marker='^', s=100, color='green', label='Buy')

            if len(sell_points) > 0:
                plt.scatter(sell_points['Date'], sell_points['Actual'], marker='v', s=100, color='red', label='Sell')

            plt.title(f'{company_name} - Trading Actions')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Plot 3: Model Predictions and Signals
            plt.subplot(3, 1, 3)
            plt.plot(results_df['Date'], results_df['Actual'], label='Actual', color='blue')
            plt.plot(results_df['Date'], results_df['Predicted'], label='Predicted', color='orange', linestyle='-')

            # Highlight buy/sell signal areas
            buy_signals = results_df[results_df['Signal'] == 'BUY']
            sell_signals = results_df[results_df['Signal'] == 'SELL']

            if len(buy_signals) > 0:
                plt.scatter(buy_signals['Date'], buy_signals['Actual'], marker='o', s=50,
                           color='green', alpha=0.5, label='Buy Signal')

            if len(sell_signals) > 0:
                plt.scatter(sell_signals['Date'], sell_signals['Actual'], marker='o', s=50,
                           color='red', alpha=0.5, label='Sell Signal')

            plt.title(f'{company_name} - Model Predictions and Signals')
            plt.xlabel('Date')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

        # Store strategy results
        strategy_results = {
            'company': company_name,
            'strategy_type': 'Model-Based',
            'strategy_df': results_df,
            'initial_capital': initial_capital,
            'final_value': final_value,
            'total_return': total_return,
            'buy_hold_return': buy_hold_return,
            'outperformance': total_return - buy_hold_return,
            'trades_count': len(buy_actions),
            'avg_holding_period': avg_holding_period,
            'parameters': {
                'buy_threshold': buy_threshold,
                'sell_threshold': sell_threshold,
                'holding_period_limit': holding_period_limit,
                'stop_loss_pct': stop_loss_pct,
                'take_profit_pct': take_profit_pct
            }
        }

        return strategy_results

    def optimize_strategy_parameters(self, company_name, model_results, initial_capital=10000,
                                    population_size=30, num_generations=20):
        """
        Optimize trading strategy parameters using genetic algorithm

        Args:
            company_name (str): Name of the company
            model_results (dict): Results from a trained model
            initial_capital (float): Initial investment amount
            population_size (int): Population size for genetic algorithm
            num_generations (int): Number of generations for genetic algorithm

        Returns:
            dict: Optimized parameters and results
        """
        print("\n" + "="*80)
        print(f"STEP 13: OPTIMIZING TRADING STRATEGY FOR {company_name}")
        print("="*80)

        # Get results DataFrame with actual and predicted prices
        results_df = model_results['results_df'].copy()

        # Define fitness function for the genetic algorithm
        def fitness_function(solution):
            """
            Enhanced fitness function for trading strategy optimization
            Higher return = higher fitness, with advanced penalties for:
            - Excessive trades (to reduce churning)
            - Long holding periods (to accelerate capital reuse)
            - Large drawdowns (to manage risk)
            - Underperforming buy & hold (to ensure market outperformance)
            """
            buy_threshold = solution[0]
            sell_threshold = solution[1]
            holding_period_limit = int(solution[2])
            stop_loss_pct = solution[3]
            take_profit_pct = solution[4]

            # Calculate predicted price change (percent)
            strategy_df = results_df.copy()
            strategy_df['Predicted_Change_Pct'] = strategy_df['Predicted'].pct_change(1).shift(-1)

            # Generate trading signals
            strategy_df['Signal'] = 'HOLD'
            strategy_df.loc[strategy_df['Predicted_Change_Pct'] > buy_threshold, 'Signal'] = 'BUY'
            strategy_df.loc[strategy_df['Predicted_Change_Pct'] < sell_threshold, 'Signal'] = 'SELL'

            # Initialize portfolio tracking
            strategy_df['Position'] = 0
            strategy_df['Capital'] = initial_capital
            strategy_df['Holdings'] = 0
            strategy_df['Portfolio'] = initial_capital
            strategy_df['Action'] = ""
            strategy_df['Days_Held'] = 0
            strategy_df['Entry_Price'] = 0

            # Trading strategy simulation
            current_position = 0
            entry_price = 0
            days_held = 0
            trades_count = 0
            total_holding_days = 0
            max_drawdown_pct = 0
            peak_value = initial_capital

            for i in range(len(strategy_df)):
                current_price = strategy_df.iloc[i]['Actual']
                current_signal = strategy_df.iloc[i]['Signal']

                # Update days held if we have a position
                if current_position > 0:
                    days_held += 1
                    strategy_df.iloc[i, strategy_df.columns.get_loc('Days_Held')] = days_held

                    # Calculate current profit/loss
                    current_return = (current_price - entry_price) / entry_price

                    # Determine if we should exit the position
                    if days_held >= holding_period_limit:
                        # Exit due to holding period limit
                        capital = strategy_df.iloc[i-1]['Capital'] + current_position * current_price
                        current_position = 0
                        total_holding_days += days_held
                        days_held = 0
                        entry_price = 0
                        trades_count += 1
                    elif current_return <= -stop_loss_pct:
                        # Exit due to stop loss
                        capital = strategy_df.iloc[i-1]['Capital'] + current_position * current_price
                        current_position = 0
                        total_holding_days += days_held
                        days_held = 0
                        entry_price = 0
                        trades_count += 1
                    elif current_return >= take_profit_pct:
                        # Exit due to take profit
                        capital = strategy_df.iloc[i-1]['Capital'] + current_position * current_price
                        current_position = 0
                        total_holding_days += days_held
                        days_held = 0
                        entry_price = 0
                        trades_count += 1
                    elif current_signal == 'SELL':
                        # Exit due to model signal
                        capital = strategy_df.iloc[i-1]['Capital'] + current_position * current_price
                        current_position = 0
                        total_holding_days += days_held
                        days_held = 0
                        entry_price = 0
                        trades_count += 1
                    else:
                        # Continue holding
                        capital = strategy_df.iloc[i-1]['Capital']
                else:  # No position
                    if current_signal == 'BUY':
                        # Calculate how many shares we can buy with available capital
                        available_capital = strategy_df.iloc[i-1]['Capital']
                        max_shares = int(available_capital / current_price)

                        if max_shares > 0:
                            # Enter new position
                            current_position = max_shares
                            entry_price = current_price
                            days_held = 0
                            capital = available_capital - (current_position * current_price)
                            trades_count += 1
                        else:
                            capital = available_capital
                    else:
                        # No action
                        capital = strategy_df.iloc[i-1]['Capital']

                # Update tracking columns
                strategy_df.iloc[i, strategy_df.columns.get_loc('Position')] = current_position
                strategy_df.iloc[i, strategy_df.columns.get_loc('Capital')] = capital
                strategy_df.iloc[i, strategy_df.columns.get_loc('Holdings')] = current_position * current_price
                portfolio_value = capital + (current_position * current_price)
                strategy_df.iloc[i, strategy_df.columns.get_loc('Portfolio')] = portfolio_value

                # Calculate running max drawdown
                if portfolio_value > peak_value:
                    peak_value = portfolio_value

                drawdown = (peak_value - portfolio_value) / peak_value
                if drawdown > max_drawdown_pct:
                    max_drawdown_pct = drawdown

            # Calculate strategy performance
            final_value = strategy_df.iloc[-1]['Portfolio']
            total_return = (final_value - initial_capital) / initial_capital * 100

            # Calculate average holding period
            avg_holding_period = total_holding_days / trades_count if trades_count > 0 else 0

            # Calculate buy and hold return
            first_price = strategy_df.iloc[0]['Actual']
            last_price = strategy_df.iloc[-1]['Actual']
            buy_hold_shares = initial_capital / first_price
            buy_hold_value = buy_hold_shares * last_price
            buy_hold_return = (buy_hold_value - initial_capital) / initial_capital * 100

            # Calculate Sharpe-like ratio (assuming risk-free rate of 0%)
            daily_returns = strategy_df['Portfolio'].pct_change().dropna()
            if len(daily_returns) > 0 and daily_returns.std() > 0:
                sharpe_ratio = (daily_returns.mean() / daily_returns.std()) * np.sqrt(252)  # Annualized
            else:
                sharpe_ratio = 0

            # Calculate fitness (higher is better)

            # Base fitness is the total return
            fitness = total_return

            # Apply penalty for excessive trading (too many trades)
            if trades_count > 10:
                fitness -= (trades_count - 10) * 0.5

            # Apply stronger penalty for too long average holding periods
            if avg_holding_period > 5:
                fitness -= (avg_holding_period - 5) * 1.5

            # Apply penalty for large drawdowns
            fitness -= max_drawdown_pct * 30  # Heavy penalty for drawdowns

            # Apply penalty for underperforming buy & hold
            if total_return < buy_hold_return:
                fitness -= (buy_hold_return - total_return) * 0.8

            # Add bonus for good Sharpe ratio
            fitness += sharpe_ratio * 2

            return fitness

        # Define parameter bounds
        param_bounds = [
            (0.001, 0.01),  # buy_threshold: 0.1% to 1%
            (-0.01, -0.001),  # sell_threshold: -1% to -0.1%
            (5, 20),  # holding_period_limit: 5 to 20 days
            (0.02, 0.1),  # stop_loss_pct: 2% to 10%
            (0.02, 0.2)  # take_profit_pct: 2% to 20%
        ]

        print(f"Optimizing trading strategy parameters using genetic algorithm")
        print(f"Population size: {population_size}, Generations: {num_generations}")
        print(f"Parameter bounds: {param_bounds}")

        # Initialize population with random solutions within bounds
        population = []
        for _ in range(population_size):
            solution = [random.uniform(low, high) for low, high in param_bounds]
            population.append(solution)

        # Genetic algorithm main loop
        best_solution = None
        best_fitness = float('-inf')

        for generation in range(num_generations):
            # Evaluate fitness for all solutions
            fitness_scores = [fitness_function(solution) for solution in population]

            # Find the best solution in this generation
            current_best_idx = np.argmax(fitness_scores)
            current_best_solution = population[current_best_idx]
            current_best_fitness = fitness_scores[current_best_idx]

            # Update overall best solution
            if current_best_fitness > best_fitness:
                best_solution = current_best_solution
                best_fitness = current_best_fitness

            # Print progress every 5 generations
            if (generation + 1) % 5 == 0:
                print(f" Generation {generation+1}/{num_generations}, Best fitness: {best_fitness:.2f}")
                print(f" Best parameters: {[round(p, 4) for p in best_solution]}")

            # Create next generation
            if generation < num_generations - 1:
                next_population = []

                # Keep the best solution (elitism)
                next_population.append(current_best_solution)

                # Selection and crossover to fill the rest of the population
                while len(next_population) < population_size:
                    # Tournament selection
                    parent1_idx = random.randint(0, population_size - 1)
                    parent2_idx = random.randint(0, population_size - 1)

                    if fitness_scores[parent1_idx] > fitness_scores[parent2_idx]:
                        parent1 = population[parent1_idx]
                    else:
                        parent1 = population[parent2_idx]

                    parent3_idx = random.randint(0, population_size - 1)
                    parent4_idx = random.randint(0, population_size - 1)

                    if fitness_scores[parent3_idx] > fitness_scores[parent4_idx]:
                        parent2 = population[parent3_idx]
                    else:
                        parent2 = population[parent4_idx]

                    # Crossover
                    crossover_point = random.randint(1, len(param_bounds) - 1)
                    child = parent1[:crossover_point] + parent2[crossover_point:]

                    # Mutation (with 20% probability)
                    if random.random() < 0.2:
                        mutation_idx = random.randint(0, len(param_bounds) - 1)
                        low, high = param_bounds[mutation_idx]
                        child[mutation_idx] = random.uniform(low, high)

                    next_population.append(child)

                population = next_population

        # Extract optimized parameters
        buy_threshold = best_solution[0]
        sell_threshold = best_solution[1]
        holding_period_limit = int(best_solution[2])
        stop_loss_pct = best_solution[3]
        take_profit_pct = best_solution[4]

        print("\nOptimized Trading Strategy Parameters:")
        print(f" Buy threshold: {buy_threshold:.4f}")
        print(f" Sell threshold: {sell_threshold:.4f}")
        print(f" Holding period limit: {holding_period_limit} days")
        print(f" Stop loss: {stop_loss_pct*100:.2f}%")
        print(f" Take profit: {take_profit_pct*100:.2f}%")

        # Test optimized strategy
        optimized_strategy = self.implement_model_based_strategy(
            company_name=company_name,
            model_results=model_results,
            initial_capital=initial_capital,
            buy_threshold=buy_threshold,
            sell_threshold=sell_threshold,
            holding_period_limit=holding_period_limit,
            stop_loss_pct=stop_loss_pct,
            take_profit_pct=take_profit_pct
        )

        # Add optimized parameters to results
        optimized_strategy['optimized_params'] = {
            'buy_threshold': buy_threshold,
            'sell_threshold': sell_threshold,
            'holding_period_limit': holding_period_limit,
            'stop_loss_pct': stop_loss_pct,
            'take_profit_pct': take_profit_pct
        }

        return optimized_strategy

    #################################
    # SECTION 9: COMPREHENSIVE PIPELINE EXECUTION
    #################################
    def run_pipeline(self, company_name=None):
        """
        Run the complete pipeline for a specific company or all companies
        Integrates all components into a cohesive workflow

        Args:
            company_name (str, optional): Name of a specific company to analyze
                If None, analyze all companies

        Returns:
            dict: Results dictionary
        """
        print("\n" + "="*80)
        print("RUNNING COMPLETE STOCK PREDICTION PIPELINE")
        print("="*80)

        # Step 1: Load data
        self.load_data()

        # Step 2: Preprocess data
        self.preprocess_data()

        # Step 3: Fill trading gaps
        self.fill_trading_gaps(method='previous')

        # Step 4: Detect and handle outliers
        self.detect_and_handle_outliers(method='zscore', window=20, threshold=3.0, handling='winsorize')

        # Step 5: Engineer features
        self.engineer_features()

        # Step 6: Split data
        split_datasets = self.split_data()

        # Define companies to analyze
        if company_name is not None:
            if company_name in self.featured_data:
                companies_to_analyze = [company_name]
            else:
                print(f"Company '{company_name}' not found in dataset. Analyzing all companies.")
                companies_to_analyze = list(self.featured_data.keys())
        else:
            companies_to_analyze = list(self.featured_data.keys())

        # Process each company
        results = {}

        for company in companies_to_analyze:
            try:
                print(f"\n{'='*50}")
                print(f"ANALYZING {company}")
                print(f"{'='*50}")

                company_results = {}

                # Get company-specific parameters
                company_params = self.market_params.get(company, {})
                seq_length = company_params.get('seq_length', self.params['seq_length'])

                # Step 7: Train LSTM model
                lstm_results = self.train_lstm_model(
                    company_name=company,
                    split_data=split_datasets[company],
                    seq_length=seq_length
                )
                company_results['LSTM'] = lstm_results

                # Step 8: Train Transformer model
                transformer_results = self.train_transformer_model(
                    company_name=company,
                    split_data=split_datasets[company],
                    seq_length=seq_length
                )
                company_results['Transformer'] = transformer_results

                # Step 9: Compare models
                comparison_results = self.compare_models(
                    company_name=company,
                    models_results=company_results
                )

                if comparison_results is None:
                    print(f"Warning: Model comparison failed for {company}, skipping to next company")
                    continue

                company_results['model_comparison'] = comparison_results

                # Get the best model
                best_model_name = comparison_results['best_model']
                best_model_results = company_results[best_model_name]

                # Step 10: Implement Bollinger Bands strategy
                bb_strategy = self.implement_bollinger_bands_strategy(
                    company_name=company,
                    data=self.featured_data[company]
                )
                company_results['bollinger_bands_strategy'] = bb_strategy

                # Step 11: Implement enhanced Bollinger Bands strategy
                enhanced_bb_strategy = self.implement_enhanced_bollinger_bands_strategy(
                    company_name=company,
                    data=self.featured_data[company]
                )
                company_results['enhanced_bb_strategy'] = enhanced_bb_strategy

                # Apply market-specific strategy if defined
                if 'strategy' in company_params and company_params['strategy'] == 'enhanced':
                    print(f"Using enhanced Bollinger Bands strategy for {company} as specified in market-specific parameters")
                    primary_strategy = enhanced_bb_strategy
                else:
                    # Step 12: Implement model-based strategy with best model
                    model_strategy = self.implement_model_based_strategy(
                        company_name=company,
                        model_results=best_model_results
                    )
                    company_results['model_strategy'] = model_strategy
                    primary_strategy = model_strategy

                # Step 13: Optimize trading strategy
                try:
                    optimized_strategy = self.optimize_strategy_parameters(
                        company_name=company,
                        model_results=best_model_results
                    )
                    company_results['optimized_strategy'] = optimized_strategy
                except Exception as e:
                    print(f"Warning: Strategy optimization failed for {company}: {str(e)}")
                    print("Using basic model strategy as a fallback")
                    company_results['optimized_strategy'] = primary_strategy  # Use primary strategy as fallback

                # Store results for this company
                results[company] = company_results

            except Exception as e:
                print(f"Error processing company {company}: {str(e)}")
                print(f"Skipping to next company")

        # Create summary of results across all companies
        summary_data = []

        for company, company_results in results.items():
            try:
                if 'model_comparison' not in company_results or 'model_strategy' not in company_results:
                    print(f"Skipping {company} in summary (incomplete results)")
                    continue

                best_model = company_results['model_comparison']['best_model']
                best_metrics = company_results['model_comparison']['best_metrics']

                # Get appropriate strategy based on company-specific parameters
                company_params = self.market_params.get(company, {})
                if 'strategy' in company_params and company_params['strategy'] == 'enhanced':
                    bb_strategy = company_results['bollinger_bands_strategy']
                    primary_strategy = company_results['enhanced_bb_strategy']
                    has_model_strategy = False
                else:
                    bb_strategy = company_results['bollinger_bands_strategy']
                    primary_strategy = company_results['model_strategy']
                    has_model_strategy = True

                # Check if optimized strategy exists, otherwise use primary strategy
                if 'optimized_strategy' in company_results and company_results['optimized_strategy'] is not None:
                    optimized_strategy = company_results['optimized_strategy']
                else:
                    print(f"Using primary strategy as optimized strategy for {company}")
                    optimized_strategy = primary_strategy

                # Create summary entry
                summary_entry = {
                    'Company': company,
                    'Best Model': best_model,
                    'MAPE (%)': best_metrics['MAPE (%)'],
                    'Directional Accuracy (%)': best_metrics['Directional Accuracy (%)'],
                    'BB Strategy Return (%)': bb_strategy['total_return'],
                    'Buy & Hold Return (%)': primary_strategy['buy_hold_return'],
                    'BB Outperformance (%)': bb_strategy['total_return'] - bb_strategy['buy_hold_return'],
                    'Optimized Strategy Return (%)': optimized_strategy['total_return'],
                    'Optimized Outperformance (%)': optimized_strategy['total_return'] - optimized_strategy['buy_hold_return']
                }

                # Add model strategy metrics if available
                if has_model_strategy:
                    summary_entry['Model Strategy Return (%)'] = primary_strategy['total_return']
                    summary_entry['Model Outperformance (%)'] = primary_strategy['total_return'] - primary_strategy['buy_hold_return']

                summary_data.append(summary_entry)

            except Exception as e:
                print(f"Error adding {company} to summary: {str(e)}")

        # Check if we have any results to summarize
        if not summary_data:
            print("No valid results to summarize")
            return {'results': results, 'summary': None}

        summary_df = pd.DataFrame(summary_data)

        print("\n" + "="*80)
        print("PIPELINE RESULTS SUMMARY")
        print("="*80)
        print(summary_df.to_string(index=False))

        # Visualize final results
        plt.figure(figsize=(14, 10))

        # Plot 1: Best Model by Company
        plt.subplot(2, 1, 1)
        model_counts = summary_df['Best Model'].value_counts()
        plt.bar(model_counts.index, model_counts.values)
        plt.title('Best Model by Count')
        plt.ylabel('Number of Companies')
        plt.grid(True, alpha=0.3, axis='y')

        # Plot 2: Strategy Performance Comparison
        plt.subplot(2, 1, 2)
        companies = summary_df['Company']
        x = range(len(companies))
        width = 0.2

        plt.bar([i - width*1.5 for i in x], summary_df['Buy & Hold Return (%)'],
                width=width, label='Buy & Hold', color='blue', alpha=0.7)
        plt.bar([i - width*0.5 for i in x], summary_df['BB Strategy Return (%)'],
                width=width, label='Bollinger Bands', color='orange', alpha=0.7)

        # Check if 'Model Strategy Return (%)' is in the summary
        if 'Model Strategy Return (%)' in summary_df.columns:
            plt.bar([i + width*0.5 for i in x], summary_df['Model Strategy Return (%)'],
                    width=width, label='Model Strategy', color='green', alpha=0.7)

        plt.bar([i + width*1.5 for i in x], summary_df['Optimized Strategy Return (%)'],
                width=width, label='Optimized Strategy', color='red', alpha=0.7)

        plt.xticks(x, companies, rotation=45)
        plt.title('Strategy Performance Comparison')
        plt.ylabel('Return (%)')
        plt.legend()
        plt.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.show()

        # Create performance comparison across markets
        self._create_market_comparison_report(summary_df)

        return {
            'results': results,
            'summary': summary_df
        }

    def _create_market_comparison_report(self, summary_df):
        """
        Create a comprehensive report comparing performance across markets
        - Addresses specific market performance
        """
        print("\n" + "="*80)
        print("MARKET COMPARISON ANALYSIS")
        print("="*80)

        # Sort markets by optimized strategy performance
        sorted_df = summary_df.sort_values('Optimized Outperformance (%)', ascending=False)

        # Print market performance ranking
        print("\nMarkets Ranked by Optimized Strategy Outperformance:")
        for i, (idx, row) in enumerate(sorted_df.iterrows()):
            print(f"{i+1}. {row['Company']}: {row['Optimized Outperformance (%)']:.2f}% outperformance")

        # Check for problematic markets (negative outperformance)
        problem_markets = sorted_df[sorted_df['Optimized Outperformance (%)'] < 0]

        if not problem_markets.empty:
            print("\nMarkets with Negative Outperformance (Potential Issues):")
            for idx, row in problem_markets.iterrows():
                print(f"- {row['Company']}: {row['Optimized Outperformance (%)']:.2f}% (vs Buy & Hold: {row['Buy & Hold Return (%)']:.2f}%)")

            print("\nRecommendations for Problematic Markets:")
            for idx, row in problem_markets.iterrows():
                best_strategy = 'Enhanced Bollinger Bands'
                if row['BB Outperformance (%)'] > row['Optimized Outperformance (%)']:
                    recommendation = f"Consider using standard Bollinger Bands strategy instead of model-based approach."
                    best_strategy = 'Standard Bollinger Bands'
                else:
                    recommendation = f"Consider adjusting model parameters or using a market-specific model architecture."

                print(f"- {row['Company']}: {recommendation}")

        # Create scatter plot comparing model accuracy vs. trading performance
        plt.figure(figsize=(12, 8))
        plt.scatter(summary_df['MAPE (%)'], summary_df['Optimized Outperformance (%)'],
                   s=100, alpha=0.7, c=summary_df['Buy & Hold Return (%)'], cmap='coolwarm')

        for i, row in summary_df.iterrows():
            plt.text(row['MAPE (%)'] + 0.1, row['Optimized Outperformance (%)'], row['Company'], fontsize=9)

        plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
        plt.xlabel('Model Accuracy (MAPE %) - Lower is Better')
        plt.ylabel('Trading Strategy Outperformance (%)')
        plt.title('Relationship Between Model Accuracy and Trading Performance')
        plt.colorbar(label='Buy & Hold Return (%)')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Overall recommendations
        print("\nOverall Recommendations:")
        print("1. For most markets, the optimized model-based trading strategy provides the best performance")
        print("2. For markets with negative outperformance, consider market-specific approaches")
        print("3. Markets with higher volatility generally benefit more from trading strategies vs. buy & hold")
        print("4. Consider adjusting the Bollinger Bands window (20-day standard, longer for trending markets)")
        print("5. Higher sequence length (6-7 days) generally improves model prediction accuracy")


# Example usage of the StockPredictionSystem
if __name__ == "__main__":
    # 1. Initialize the system with the Excel data file
    file_path = "/content/2020Q1Q2Q3Q4-2021Q1.xlsx"  # Update with actual file path
    stock_system = StockPredictionSystem(file_path)

    # 2. Run the full pipeline for all companies
    results = stock_system.run_pipeline()

    # 3. Alternatively, run for a specific company
    # results = stock_system.run_pipeline(company_name="South Africa - Impala Platinum")

    # 4. Print detailed analysis of market-specific performance
    print("\n" + "="*80)
    print("DETAILED MARKET-SPECIFIC ANALYSIS")
    print("="*80)

    if results['summary'] is not None:
        for company in results['summary']['Company'].unique():
            company_data = results['summary'][results['summary']['Company'] == company].iloc[0]
            print(f"\nAnalysis for {company}:")
            print(f"  Best Model: {company_data['Best Model']}")
            print(f"  MAPE: {company_data['MAPE (%)']:.2f}%")
            print(f"  Directional Accuracy: {company_data['Directional Accuracy (%)']:.2f}%")
            print(f"  Optimized Strategy Return: {company_data['Optimized Strategy Return (%)']:.2f}%")
            print(f"  Buy & Hold Return: {company_data['Buy & Hold Return (%)']:.2f}%")
            print(f"  Outperformance: {company_data['Optimized Outperformance (%)']:.2f}%")

            if company_data['Optimized Outperformance (%)'] < 0:
                print(f"  ** WARNING: Strategy underperforms Buy & Hold **")
                print(f"  Recommendation: Consider market-specific parameters or alternative strategy")

    # 5. Print summary recommendations
    print("\n" + "="*80)
    print("OVERALL RECOMMENDATIONS")
    print("="*80)

    print("""
1. **Bollinger Bands Configuration**:
   - Increase the window from 5 to 20 days for most markets
   - For trending markets, consider 30-day window
   - This prevents bands from reacting too quickly to short-term fluctuations

2. **LSTM Sequence Length**:
   - Increase from 5 to 6-7 days as suggested
   - Longer sequence length provides better context for predictions
   - Helps address underprediction issues in some markets

3. **Market-Specific Approaches**:
   - South Africa market requires special attention
   - Use enhanced Bollinger Bands strategy for South Africa instead of model-based
   - Consider other market-specific parameters for underperforming markets

4. **Volume Impact Analysis**:
   - Add multiple timeframes for volume analysis (5, 10, 20 days)
   - Include volume-price correlation features
   - Consider volume breakout detection for entry/exit signals

5. **Model Documentation**:
   - Each strategy has been documented with clear explanations
   - Added comprehensive performance comparison across markets
   - Included detailed analysis of when and why models fail

6. **Hybrid Approach**:
   - Use optimized model-based strategy for most markets
   - Switch to alternative strategies for specific markets when needed
   - This balances performance with maintainability
""")